# X-ray focusing with compound refractive lenses

1. Run **1 · Prepare XPOSim**.
2. Run **2 · CRL and source parameters**. Set the source, compound refractive lens (CRL) geometry and material. The lens element cross-section sits next to its geometry fields. The optical layout and analytical estimates appear below the inputs.
3. Run **3 · Results**, then press **Compute**. No optical calculation starts before that click. To change the lens or source later, edit the fields in cell 2 and press **Compute** again in cell 3.

After a successful calculation, use **Download PDF report** at the bottom of Results. The one-page A4 report includes the optical layout and axial scan. For crossed CRLs it compares the transverse sections at the paraxial image plane and the equal-FWHM plane. For a single CRL it shows sections around the numerical reference plane.

**Run all** initializes both panels without calculating. For GPU: **Runtime → Change runtime type → GPU**, reconnect, and run all. The actual CUDA device is checked.


In [ ]:
#@title 1 · Prepare XPOSim
import hashlib
import importlib.util
import importlib.metadata
import subprocess
import sys
import urllib.request
from pathlib import Path

SOURCE_REF = 'cd18ad1b412133729b3076ef581d0e167bd1ea82'
SOURCE_SHA256 = '4053132863acbe4b2140ebcc8ebbf89a08b239a5e592bfe6a0962d9fddf74199'
PROJECT_DIR = Path.cwd() / '.xposim_runtime' / SOURCE_REF
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
requirements = {'numpy':'numpy', 'scipy':'scipy', 'matplotlib':'matplotlib', 'ipywidgets':'ipywidgets>=8,<9'}
missing = [requirement for name,requirement in requirements.items() if importlib.util.find_spec(name) is None]
if importlib.util.find_spec('ipywidgets') is not None:
    try:
        widget_major = int(importlib.metadata.version('ipywidgets').split('.')[0])
    except (importlib.metadata.PackageNotFoundError, ValueError):
        widget_major = None
    if widget_major != 8:
        missing.append(requirements['ipywidgets'])
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-cache-dir',*missing])
for relative in ['XPOSim.py','xray_material_data/AtomicConstants.dat',
                 'xray_material_data/CrossSec-Compton_McMaster.dat','xray_material_data/f1f2_Windt.dat']:
    target = PROJECT_DIR / relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if not target.exists():
        url = f'https://raw.githubusercontent.com/mamichberdey/XPOSim/{SOURCE_REF}/{relative}'
        with urllib.request.urlopen(url,timeout=90) as response:
            content = response.read()
        temporary = target.with_suffix(target.suffix+'.download')
        temporary.write_bytes(content)
        temporary.replace(target)
if hashlib.sha256((PROJECT_DIR/'XPOSim.py').read_bytes()).hexdigest()!=SOURCE_SHA256:
    raise RuntimeError('XPOSim checksum mismatch. Remove .xposim_runtime and repeat preparation.')
spec = importlib.util.spec_from_file_location('XPOSim',PROJECT_DIR/'XPOSim.py')
xpos = importlib.util.module_from_spec(spec)
sys.modules['XPOSim'] = xpos
spec.loader.exec_module(xpos)
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
print(f'XPOSim {xpos.__version__}; available backend: {xpos.select_backend("auto")}.')
print('Ready. Run cell 2 to open CRL and source inputs, then cell 3 for Results. Press Compute only after editing the inputs.')


In [ ]:
#@title 2 · CRL and source parameters — run this cell to show the input fields
import html
import math
import ipywidgets as widgets
from traitlets import TraitError
from IPython.display import display
try:
    from google.colab import output as colab_output
except ImportError:
    colab_output=None
if colab_output is not None:
    colab_output.enable_custom_widget_manager()

def _detect_theme():
    if colab_output is not None:
        try:
            detected=colab_output.eval_js("""(() => {
              const root=document.documentElement, body=document.body;
              const attr=(root.getAttribute('theme')||body.getAttribute('theme')||'').toLowerCase();
              if(attr==='dark'||attr==='light')return attr;
              if(root.classList.contains('dark')||body.classList.contains('dark'))return 'dark';
              const css=getComputedStyle(body);
              const primary=css.getPropertyValue('--colab-primary-text-color').trim();
              const probe=document.createElement('span');probe.style.color=primary||css.color;body.appendChild(probe);
              const resolved=getComputedStyle(probe).color;probe.remove();
              const colors=resolved.match(/[0-9.]+/g);
              if(colors&&colors.length>=3){
                const luminance=.2126*Number(colors[0])+.7152*Number(colors[1])+.0722*Number(colors[2]);
                return luminance>150?'dark':'light';
              }
              return matchMedia('(prefers-color-scheme:dark)').matches?'dark':'light';
            })()""",timeout_sec=2)
            if detected in ('light','dark'):return detected
        except Exception:
            pass
    return globals().get('current_theme','light')


if 'current_theme' not in globals():
    current_theme=_detect_theme()


_previous_values={name:widget.value for name,widget in globals().get('controls',{}).items()}
_previous_values['accuracy']={'Quick':'Fast','Refined':'Accurate'}.get(_previous_values.get('accuracy'),_previous_values.get('accuracy','Fast'))
for key,labels in (
    ('focusing',{'One-dimensional focusing':'one-dimensional','Two-dimensional focusing':'two-dimensional'}),
    ('source',{'Point source':'point','Gaussian source (incoherent)':'gaussian','Plane wave at CRL entrance':'plane'}),
    ('source_method',{'Gaussian convolution':'convolution','Incoherent point sampling':'points'}),
):
    if key in _previous_values:
        _previous_values[key]=labels.get(_previous_values[key],_previous_values[key])
if 'material_mode' not in _previous_values:
    _previous_values['material_mode']='Chemical formula' if _previous_values.get('material')=='Custom' else 'Preset material'
if 'mesh_mode' not in _previous_values:
    _previous_values['mesh_mode']='Set point counts' if _previous_values.get('nx',0) or _previous_values.get('ny',0) else 'Auto'
if 'roi_mode' not in _previous_values:
    _previous_values['roi_mode']='Manual' if _previous_values.get('z_mode')=='Manual bounds' else 'Auto'
_first_explorer_initialization=not bool(globals().get('controls'))
_material_state=globals().get('_material_state',{
    'manual_density':globals().get('_manual_density_value',_previous_values.get('density',2.33) if _previous_values.get('density_mode')=='Manual' else 2.33),
    'density_edited':globals().get('_manual_density_user_set',_previous_values.get('density_mode')=='Manual'),
    'preset_density_mode':globals().get('_preset_density_mode',_previous_values.get('density_mode','Auto')),
    'coefficients_edited':globals().get('_coefficients_initialized',False),
})
_y_material_state=globals().get('_y_material_state',{
    'manual_density':2.33,'density_edited':False,'preset_density_mode':'Auto','coefficients_edited':False,
})
_show_detector=getattr(globals().get('detector_toggle'),'value',False)
_previous_editor=globals().get('parameter_editor')
if _previous_editor is not None:_previous_editor.close()
for widget in globals().get('controls',{}).values():
    widget.unobserve_all();widget.close()
controls={};containers={};field_labels={}

def input_field(name,label,widget,tooltip=''):
    if name in _previous_values:
        try:
            value=_previous_values[name]
            widget.value=value
        except (ValueError,TypeError,TraitError):pass
    controls[name]=widget
    widget.layout.width='100%'
    widget.layout.height='29px'
    caption=widgets.HTML('<span style="font-size:14px;font-weight:500;line-height:1.2" title="'+html.escape(tooltip,quote=True)+'">'+html.escape(label)+'</span>',layout=widgets.Layout(height='auto',min_height='27px',flex='1 0 auto',width='100%',overflow='visible',margin='0'))
    field_labels[name]=caption
    box=widgets.VBox([caption,widget],layout=widgets.Layout(width='100%',min_width='0',height='100%'))
    box.add_class('xposim-input-field')
    caption.add_class('xposim-field-caption')
    box.add_class('xposim-field-'+name)
    containers[name]=box
    return box

def group(fields,columns=4):
    return widgets.GridBox(fields,layout=widgets.Layout(
        grid_template_columns=f'repeat(auto-fit,minmax(min(100%,max({260 if columns==4 else 220}px,calc((100% - {14*(columns-1)}px) / {columns}))),1fr))',grid_gap='7px 12px',width='100%'))

def section_heading(title):
    return '<div style="font-size:16px;font-weight:650;letter-spacing:.01em;padding:1px 0 5px">'+html.escape(title)+'</div>'

def section(title,children,kind):
    parts=children
    if title:
        heading=widgets.HTML(section_heading(title))
        parts=[heading,*children]
    box=widgets.VBox(parts,layout=widgets.Layout(width='100%',min_width='0',padding='8px 12px',border='1px solid #d5dde5',margin='0'))
    box.add_class('xposim-section');box.add_class('xposim-section-'+kind)
    return box

source_controls=group([
    input_field('source','Source model',widgets.Dropdown(options=[('Point source','point'),
        ('Incoherent Gaussian source','gaussian'),('Plane wave','plane')])),
    input_field('energy','Photon energy (keV)',widgets.FloatText(value=12.0)),
    input_field('distance','Source-to-CRL distance (m)',widgets.FloatText(value=15.0)),
    input_field('source_method','Extended-source method',widgets.Dropdown(options=[('Intensity convolution','convolution'),
        ('Incoherent point sampling','points')])),
    input_field('fwhm_x','Source FWHM along x (µm)',widgets.FloatText(value=100.0)),
    input_field('fwhm_y','Source FWHM along y (µm)',widgets.FloatText(value=1000.0)),
    input_field('source_points','Source points per axis',widgets.BoundedIntText(value=7,min=3,max=101,step=2)),
    input_field('source_points_y','Source points along y',widgets.BoundedIntText(value=21,min=3,max=101,step=2),
                'Independent sampling along the y axis for crossed CRLs.'),
])
general_controls=group([
    input_field('focusing','CRL focusing',widgets.Dropdown(options=[('Line focus (1D)','one-dimensional'),
        ('Point focus (2D)','two-dimensional'),('Crossed CRLs (2D)','crossed')])),
    input_field('grid','Calculation grid',widgets.Dropdown(options=['1D','2D'])),
    input_field('accuracy','Sampling',widgets.Dropdown(options=['Fast','Accurate'],value='Fast')),
    widgets.VBox([
        input_field('elements','Number of lens elements N',widgets.BoundedIntText(value=26,min=1,max=10000)),
        input_field('crossed_x_elements','Number of lens elements N',widgets.BoundedIntText(value=6,min=1,max=10000)),
    ],layout=widgets.Layout(width='100%',min_width='0')),
])
def geometry_fields(prefix='',unfocused_axis='y'):
    return [
        input_field(prefix+'radius','Radius of curvature R (µm)',widgets.FloatText(value=6.25),
                    'Radius of curvature at the apex of each parabolic surface.'),
        input_field(prefix+'aperture','Geometric aperture A (µm)',widgets.FloatText(value=50.0),
                    'Full width for a line-focus lens or diameter for a point-focus lens.'),
        input_field(prefix+'bridge','Web thickness d (µm)',widgets.FloatText(value=2.0),
                    'Minimum material thickness between the two lens surfaces.'),
        input_field(prefix+'depth','Structure depth L (µm)',widgets.FloatText(value=30.0),
                    f'Extent of a line-focus lens along the unfocused {unfocused_axis} axis.'),
    ]

geometry_controls=group(geometry_fields(),2)
def material_fields(prefix=''):
    names=[prefix+name for name in ('material_mode','material','formula','density','density_mode','delta','beta')]
    mode,material,formula,density,density_mode,delta,beta=names
    mode_field=input_field(mode,'Material definition',widgets.Dropdown(options=['Preset material','Chemical formula','Direct δ / β']))
    material_field=input_field(material,'Preset material',widgets.Dropdown(
        options=['Beryllium (Be)','Silicon (Si)','Aluminium (Al)','Diamond (C)'],value='Silicon (Si)'))
    formula_field=input_field(formula,'Chemical formula',widgets.Text(value='SiO2',continuous_update=False))
    density_field=input_field(density,'Density (g/cm³)',widgets.FloatText(value=2.33))
    density_choice=widgets.Dropdown(options=['Auto','Manual'],value=_previous_values.get(density_mode,'Auto'),
        layout=widgets.Layout(width='94px',height='26px'))
    density_choice.add_class('xposim-field-'+density_mode)
    controls[density_mode]=density_choice
    field_labels[density].layout.width='auto'
    density_field.children=(widgets.HBox([field_labels[density],density_choice],
        layout=widgets.Layout(width='100%',min_height='27px',flex_flow='row wrap',justify_content='space-between',
                              align_items='center')),controls[density])
    return group([mode_field,material_field,formula_field,density_field,
        input_field(delta,'Manual δ',widgets.Text(value='3.4e-6',continuous_update=False),
            'Dimensionless refractive index decrement. Scientific notation is accepted.'),
        input_field(beta,'Manual β',widgets.Text(value='3e-8',continuous_update=False),
            'Dimensionless absorption index. It may be zero.'),
    ],2)

material_controls=material_fields()
second_geometry_controls=group([
    input_field('y_elements','Number of lens elements N',widgets.BoundedIntText(value=58,min=1,max=10000)),
    *geometry_fields('y_',unfocused_axis='x'),
],2)
second_material_controls=material_fields('y_')
spacing_controls=group([
    input_field('gap_mode','Stack spacing',widgets.Dropdown(options=['Auto-align','Manual'])),
    input_field('gap_mm','Manual inter-stack gap (mm)',widgets.FloatText(value=0.0),
        'Free space from the x CRL exit to the y CRL entrance. Zero places the stacks in contact.'),
    input_field('alignment_tol_um','Axial alignment tolerance (µm)',widgets.FloatText(value=1.0),
        'Maximum allowed separation of the two paraxial image planes.'),
],2)
advanced_fields=group([
    input_field('backend','Compute backend',widgets.Dropdown(options=[
        ('Auto','auto'), ('NumPy (CPU)','numpy'), ('CuPy (CUDA)','cupy'), ('PyTorch (CUDA)','torch'), ('JAX (GPU)','jax')])),
    input_field('mesh_mode','Transverse grid control',widgets.Dropdown(options=['Auto','Set spacing','Set point counts'])),
    input_field('dx_nm','Target propagation Δx (nm)',widgets.FloatText(value=30.0),'Actual spacings are rounded to a scale-adaptive multiple of five and reported beside the backend.'),
    input_field('dy_nm','Target propagation Δy (nm)',widgets.FloatText(value=30.0)),
    input_field('z_spacing_mode','Axial grid control',widgets.Dropdown(options=['Auto','Set spacing'])),
    input_field('dz_mm','Target Δz (mm)',widgets.FloatText(value=.05),'Preferred axial steps use 1, 2.5 or 5 times a power of ten; the actual value is reported after calculation.'),
    input_field('adaptive_window','Adapt propagation window',widgets.Checkbox(value=True,indent=False)),
    input_field('nx','Grid points x (0 = auto)',widgets.IntText(value=0)),
    input_field('ny','Grid points y (0 = auto)',widgets.IntText(value=0)),
    input_field('window','Initial window / geometry',widgets.FloatText(value=1.3)),
    input_field('batch','Batch size (0 = auto)',widgets.IntText(value=0)),
    input_field('z_points','Target z sample count',widgets.BoundedIntText(value=51,min=5,max=501)),
    input_field('z_level','Axial ROI intensity fraction',widgets.BoundedFloatText(value=.2,min=.001,max=.99,step=.05)),
    input_field('xy_level','Transverse ROI intensity fraction',widgets.BoundedFloatText(value=.01,min=.00001,max=.5,step=.01)),
    input_field('tolerance','Peak z tolerance (µm; 0 = auto)',widgets.FloatText(value=0.0)),
])
roi_fields=group([
    input_field('roi_mode','Region selection',widgets.Dropdown(options=['Auto','Manual'])),
    input_field('z_min','Scan z minimum (mm)',widgets.FloatText(value=25.0)),
    input_field('z_max','Scan z maximum (mm)',widgets.FloatText(value=45.0)),
    input_field('x_min','x minimum (µm)',widgets.FloatText(value=-1.0)),
    input_field('x_max','x maximum (µm)',widgets.FloatText(value=1.0)),
    input_field('y_min','y minimum (µm)',widgets.FloatText(value=-1.0)),
    input_field('y_max','y maximum (µm)',widgets.FloatText(value=1.0)),
],4)
roi_status=widgets.HTML(layout=widgets.Layout(width='100%',min_height='30px'))
optical_layout=widgets.HTML(layout=widgets.Layout(width='100%'))
element_preview=widgets.HTML(layout=widgets.Layout(width='100%'))
second_element_preview=widgets.HTML(layout=widgets.Layout(width='100%'))

PRESETS={'Beryllium (Be)':('Be',1.848),'Silicon (Si)':('Si',2.33),'Aluminium (Al)':('Al',2.70),'Diamond (C)':('C',3.515)}
_syncing_controls=False
_material_state['last_mode']=controls['material_mode'].value
_y_material_state['last_mode']=controls['y_material_mode'].value
_preflight_result=None
preflight_geometry=widgets.HTML(layout=widgets.Layout(width='100%'))
preflight_material=widgets.HTML(layout=widgets.Layout(width='100%'))
second_preflight_material=widgets.HTML(layout=widgets.Layout(width='100%'))
preflight_status=widgets.HTML(layout=widgets.Layout(width='100%',min_height='23px',overflow='visible',margin='0'))
preflight_geometry.add_class('xposim-analytic-geometry')
preflight_material.add_class('xposim-analytic-material')
preflight_status.add_class('xposim-analytic-status')


def sync_material_fields(prefix,state,analytical,change):
    widgets_by_name={name:controls[prefix+name] for name in
                     ('material_mode','material','formula','density','density_mode','delta','beta')}
    mode=widgets_by_name['material_mode'].value
    preset=mode=='Preset material'
    direct=mode=='Direct δ / β'
    density=widgets_by_name['density']
    choice=widgets_by_name['density_mode']
    delta,beta=widgets_by_name['delta'],widgets_by_name['beta']
    if change and change['owner'] is choice and change['new']=='Manual' and preset and not state['density_edited']:
        state['manual_density']=density.value
    if change and change['owner'] is density and not density.disabled:
        state['manual_density']=change['new']
        state['density_edited']=True
    if change and change['owner'] in (delta,beta) and not delta.disabled:
        state['coefficients_edited']=True
    if mode!=state['last_mode']:
        if state['last_mode']=='Preset material':
            state['preset_density_mode']=choice.value
        choice.value=state['preset_density_mode'] if preset else 'Manual'
        if direct and not state['coefficients_edited'] and analytical is not None:
            delta.value=f'{analytical.delta:.10g}'
            beta.value=f'{analytical.beta:.10g}'
            state['coefficients_edited']=True
    state['last_mode']=mode
    if preset:
        state['preset_density_mode']=choice.value
    else:
        choice.value='Manual'
    widgets_by_name['material'].disabled=not preset
    widgets_by_name['formula'].disabled=mode!='Chemical formula'
    delta.disabled=beta.disabled=not direct
    choice.disabled=not preset
    automatic=preset and choice.value=='Auto'
    density.disabled=direct or automatic
    if automatic:
        density.value=PRESETS[widgets_by_name['material'].value][1]
    elif not direct:
        density.value=state['manual_density']

def sync_controls(change=None):
    global _syncing_controls
    if _syncing_controls:return
    _syncing_controls=True
    try:
        crossed=controls['focusing'].value=='crossed'
        previous_pair=isinstance(_preflight_result,xpos.CrossedCRLAnalysis)
        sync_material_fields('',_material_state,_preflight_result.x_lens if previous_pair else _preflight_result,change)
        if crossed:
            sync_material_fields('y_',_y_material_state,_preflight_result.y_lens if previous_pair else None,change)
        two_focusing=controls['focusing'].value!='one-dimensional'
        if two_focusing:controls['grid'].value='2D'
        controls['grid'].disabled=two_focusing
        controls['gap_mm'].disabled=controls['gap_mode'].value!='Manual'
        containers['elements'].layout.display='none' if crossed else ''
        containers['crossed_x_elements'].layout.display='' if crossed else 'none'
        if 'second_stack_section' in globals():
            second_stack_section.layout.display='' if crossed else 'none'
            spacing_section.layout.display='' if crossed else 'none'
            geometry_section.children[0].value=section_heading('x-focusing CRL geometry' if crossed else 'CRL geometry')
            material_section.children[0].value=section_heading('x-focusing CRL material' if crossed else 'Material')
        two_grid=controls['grid'].value=='2D'
        gaussian=controls['source'].value=='gaussian'
        point_sampling=gaussian and controls['source_method'].value=='points'
        containers['source_points_y'].layout.display='' if crossed and point_sampling else 'none'
        count_caption=field_labels['source_points'].value
        old,new=('Source points per axis','Source points along x') if crossed else ('Source points along x','Source points per axis')
        field_labels['source_points'].value=count_caption.replace(old,new)
        manual_roi=controls['roi_mode'].value=='Manual'
        active={'distance':controls['source'].value!='plane',
            'source_method':gaussian,'fwhm_x':gaussian,'fwhm_y':gaussian and two_grid,
            'source_points':point_sampling,'source_points_y':crossed and point_sampling,
            'depth':two_grid and controls['focusing'].value!='two-dimensional','ny':two_grid and controls['mesh_mode'].value=='Set point counts',
            'nx':controls['mesh_mode'].value=='Set point counts',
            'dx_nm':controls['mesh_mode'].value=='Set spacing','dy_nm':two_grid and controls['mesh_mode'].value=='Set spacing',
            'dz_mm':controls['z_spacing_mode'].value=='Set spacing','z_points':controls['z_spacing_mode'].value=='Auto',
            'z_min':manual_roi,'z_max':manual_roi,'x_min':manual_roi,'x_max':manual_roi,
            'y_min':manual_roi and two_grid,'y_max':manual_roi and two_grid,
            'z_level':not manual_roi,'xy_level':not manual_roi}
        for name,enabled in active.items():controls[name].disabled=not enabled
    finally:_syncing_controls=False


def detector_bounds(v):
    if v['roi_mode']=='Auto':
        return None,None
    z=(v['z_min']*1e-3,v['z_max']*1e-3)
    x=(v['x_min']*1e-6,v['x_max']*1e-6)
    y=(v['y_min']*1e-6,v['y_max']*1e-6) if v['grid']=='2D' else None
    for axis,bounds in [('z',z),('x',x),*([('y',y)] if y is not None else [])]:
        if not all(map(math.isfinite,bounds)) or bounds[0]>=bounds[1]:
            raise ValueError(f'{axis} minimum must be finite and below its maximum.')
    if z[0]<0:
        raise ValueError('z minimum must be zero or positive.')
    return z,(x,y) if y is not None else x


def optical_stack_values(v,prefix=''):
    mode=v[prefix+'material_mode']
    if mode=='Direct δ / β':
        try:
            delta,beta=float(v[prefix+'delta']),float(v[prefix+'beta'])
        except ValueError:
            which='y CRL' if prefix else 'CRL'
            raise ValueError(f'Enter δ and β for the {which} as numbers, for example 1.23e-6.') from None
        formula=density=None
    else:
        delta=beta=None
        if mode=='Preset material':
            formula,preset_density=PRESETS[v[prefix+'material']]
            density=preset_density if v[prefix+'density_mode']=='Auto' else v[prefix+'density']
        else:
            formula,density=v[prefix+'formula'],v[prefix+'density']
    return dict(radius_m=v[prefix+'radius']*1e-6,aperture_m=v[prefix+'aperture']*1e-6,
                bridge_thickness_m=v[prefix+'bridge']*1e-6,n_elements=v[prefix+'elements'],
                structure_depth_m=v[prefix+'depth']*1e-6,material_formula=formula,density_g_cm3=density,
                material_delta=delta,material_beta=beta)

def read_configuration(*,include_detector=True):
    v={name:widget.value for name,widget in controls.items()}
    scan_bounds,roi=detector_bounds(v) if include_detector else (None,None)
    names=['batch','tolerance']+(['nx']+(['ny'] if v['grid']=='2D' else []) if v['mesh_mode']=='Set point counts' else [])
    for name in names:
        if v[name]<0:raise ValueError(f'{name} must be non-negative; zero selects automatic control.')
    crossed=v['focusing']=='crossed'
    gaussian=v['source']=='gaussian'
    spacing=dict(second_crl=xpos.CRLStackConfig(**optical_stack_values(v,'y_')),
                 gap_m=v['gap_mm']*1e-3 if v['gap_mode']=='Manual' else None,
                 alignment_tolerance_m=v['alignment_tol_um']*1e-6) if crossed else {}
    first_stack=optical_stack_values(v)
    if crossed:
        first_stack['n_elements']=v['crossed_x_elements']
    return xpos.CRLFocusConfig(
        **first_stack,**spacing,energy_kev=v['energy'],
        geometry=v['focusing'],grid_dimension=2 if v['grid']=='2D' else 1,
        source_distance_m=None if v['source']=='plane' else v['distance'],
        source_type=v['source'],
        source_fwhm_x_m=v['fwhm_x']*1e-6 if gaussian else 0.,source_fwhm_y_m=v['fwhm_y']*1e-6 if gaussian and v['grid']=='2D' else 0.,
        source_method=v['source_method'],source_points=v['source_points'],backend=v['backend'],
        source_points_y=v['source_points_y'] if crossed and gaussian and v['source_method']=='points' else None,
        accuracy='preview' if v['accuracy']=='Fast' else 'refined',
        grid_points=(v['nx'] or None) if v['mesh_mode']=='Set point counts' else None,
        grid_points_y=(v['ny'] or None) if v['grid']=='2D' and v['mesh_mode']=='Set point counts' else None,
        grid_step_x_m=v['dx_nm']*1e-9 if v['mesh_mode']=='Set spacing' else None,
        grid_step_y_m=v['dy_nm']*1e-9 if v['grid']=='2D' and v['mesh_mode']=='Set spacing' else None,
        scan_step_m=v['dz_mm']*1e-3 if v['z_spacing_mode']=='Set spacing' else None,
        window_factor=v['window'],adaptive_window=v['adaptive_window'],batch_size=v['batch'] or None,scan_points=v['z_points'],
        scan_range_m=scan_bounds,detector_roi_m=roi,
        axial_roi_level=v['z_level'],transverse_roi_level=v['xy_level'],peak_search_tolerance_m=v['tolerance']*1e-6 if v['tolerance'] else None)


def decimal_number(value,digits=2):
    text=f'{value:.{digits}f}'.rstrip('0').rstrip('.')
    return '0' if text=='-0' else text


def analytical_number(value,scientific=False):
    if value is None:return '—'
    if not math.isfinite(value):return '∞' if value>0 else '−∞'
    if scientific and value:
        mantissa,exponent=f'{value:.4e}'.split('e')
        return mantissa.rstrip('0').rstrip('.')+' × 10<sup>'+str(int(exponent))+'</sup>'
    return decimal_number(value)


def analytical_length(value):
    if value is None:return '—'
    if not math.isfinite(value):return analytical_number(value)
    scale,unit=(1.,'m') if abs(value)>=.1 else ((1e-3,'mm') if abs(value)>=1e-4 else (1e-6,'µm'))
    return analytical_number(value/scale)+' '+unit


def analytical_entry(label,value,*,css_class=''):
    attribute=f' class="{css_class}"' if css_class else ''
    return '<div'+attribute+'><dt>'+html.escape(label)+'</dt><dd>'+value+'</dd></div>'


def magnification_row(axis,value):
    return (f'<div class="xposim-magnification-axis"><span class="xposim-axis-name">{axis}</span>'
            f'<span class="xposim-number">{analytical_number(value,True)}</span></div>')


def coherence_values(values,formatter):
    labels=[formatter(value) for value in values]
    if len(labels)==1 or labels[0]==labels[1]:
        return labels[0]
    return 'x '+labels[0]+' · y '+labels[1]


def schematic_extent(value, reference, minimum, maximum):
    """Map a physical dimension monotonically into the illustrated range."""
    return minimum+(maximum-minimum)*value/(value+reference)


def schematic_material(cfg):
    return 'Material '+cfg.material_formula if cfg.material_delta is None and cfg.material_formula is not None else 'Direct δ / β'


class Schematic:
    """Draw the same geometry in the notebook SVG and the printed report."""
    print_style={
        'xp-layout-axis':('#8293a6',None,.7,'--'),
        'xp-layout-dimension':('#516279',None,1,'-'),
        'xp-layout-lens':('#4d6788','#d9e5f3',1,'-'),
        'xp-radius-marker':('#627b9c',None,1.05,'-'),
        'xp-dimension-arrow':('#516279','#516279',.6,'-'),
        'xp-round-back':('none','#8ea5b9',0.,'-'),
        'xp-round-cap':('#566d81','#8399ad',1.,'-'),
        'xp-round-aperture':('#566d81','#c5d4e0',.8,'-'),
        'xp-round-case-top':('none','#b7c8d5',0.,'-'),
        'xp-round-case-light':('none','#a9bccb',0.,'-'),
        'xp-round-case-mid':('none','#9aafc1',0.,'-'),
        'xp-round-case-bottom':('none','#8199ae',0.,'-'),
        'xp-round-cut':('#829ab0','#d4e0ea',.65,'-'),
        'xp-round-window-edge':('#5e768b',None,.9,'-'),
        'xp-round-internal-ray':('#b7782e',None,.8,'--'),
        'xp-round-cavity-near':('#667d96','#b5c5d8',.65,'-'),
        'xp-round-cavity-far':('#667d96','#8fa6bf',.65,'-'),
        'xp-round-mesh':('#526e8b',None,.62,'-'),
        'xp-layout-slab':('#516279','#e7edf4',.9,'-'),
        'xp-layout-cavity':('#516279',None,.85,'-'),
        'xp-layout-hidden':('#9aa9ba',None,.7,'--'),
        'xp-layout-source-shape':('#a66798','#f9ebdd',.9,'-'),
        'xp-layout-ray':('#b7782e',None,1,'-'),
        'xp-layout-source':('#9d651f','#e5ae61',.8,'-'),
        'xp-layout-focus':('#12786e','#12786e',.9,'-'),
        'xp-layout-focus-line':('#12786e',None,1.5,'-'),
        'xp-crossed-x':('#217e94',None,1,'-'),
        'xp-crossed-y':('#a86440',None,1,'-'),
    }

    def __init__(self,width,height,ax=None):
        self.width,self.height=width,height
        self.ax=ax
        self.parts=[]
        if ax is not None:
            ax.set(xlim=(0,width),ylim=(height,0),aspect='equal')
            ax.set_axis_off()
            fig=ax.figure
            box=ax.get_position()
            self.print_scale=min(box.width*fig.get_figwidth()*72/width,box.height*fig.get_figheight()*72/height)

    def path(self,points,classes,*,close=False):
        self.compound((points,),classes,close=close)

    def compound(self,segments,classes,*,close=True):
        if self.ax is None:
            d=' '.join('M '+' L '.join(f'{x:.7g} {y:.7g}' for x,y in points)+(' Z' if close else '')
                       for points in segments)
            self.parts.append(f'<path class="{classes}" fill-rule="evenodd" d="{d}"/>')
            return
        from matplotlib.path import Path as PlotPath
        from matplotlib.patches import PathPatch
        stroke,fill,width,dash=self.print_style[classes.split()[0]]
        paths=[]
        orientation=None
        for points in segments:
            vertices=np.asarray(points)
            if close and len(segments)>1:
                x,y=vertices.T
                area=np.dot(x,np.roll(y,-1))-np.dot(y,np.roll(x,-1))
                if orientation is None:orientation=area
                elif area*orientation>0:vertices=vertices[::-1]
            paths.append(PlotPath(np.vstack((vertices,vertices[0])),closed=True) if close else PlotPath(vertices))
        self.ax.add_patch(PathPatch(PlotPath.make_compound_path(*paths),facecolor=fill or 'none',
            edgecolor=stroke,linewidth=width*.72,linestyle=dash,zorder=2))

    def label(self,x,y,text,classes='xp-layout-label',anchor='middle'):
        if self.ax is None:
            self.parts.append(f'<text class="{classes}" x="{x:.4g}" y="{y:.4g}" text-anchor="{anchor}">{html.escape(text)}</text>')
            return
        small=classes.startswith('xp-layout-small')
        size=max(5.8,min(10,(14 if small else 17)*self.print_scale))
        self.ax.text(x,y,text,ha={'start':'left','middle':'center','end':'right'}[anchor],
            va='baseline',fontsize=size,color='#41546b' if small else '#172235',
            fontweight='normal' if small else 'semibold',zorder=4,usetex=False,parse_math=False)

    def dot(self,x,y,radius,classes):
        if self.ax is None:
            self.parts.append(f'<circle class="{classes}" cx="{x:.5g}" cy="{y:.5g}" r="{radius}"/>')
            return
        from matplotlib.patches import Circle
        stroke,fill,lw,_=self.print_style[classes]
        self.ax.add_patch(Circle((x,y),radius,facecolor=fill,edgecolor=stroke,linewidth=lw*.72,zorder=3))

    def ruler(self,start,stop,offset,classes=''):
        """Offset a dimension line and draw the two measured endpoints."""
        a,b,shift=np.asarray(start),np.asarray(stop),np.asarray(offset)
        delta=b-a
        perpendicular=np.array([-delta[1],delta[0]])/np.linalg.norm(delta)
        style='xp-layout-dimension '+classes
        p,q=a+shift,b+shift
        extension=shift*(4/max(4,np.linalg.norm(shift)))
        self.path([a,p+extension],style)
        self.path([b,q+extension],style)
        self.path([p,q],style)
        for end in (p,q):
            self.path([end-3*perpendicular,end+3*perpendicular],style)
        return (p+q)/2

    def html(self,css,description,defs=''):
        return (f'<div class="{css}"><svg role="img" aria-label="{description}" viewBox="0 0 {self.width} {self.height}">'
                +defs+''.join(self.parts)+'</svg></div>')


def element_schematic(cfg,result,*,ax=None,show_values=True,show_depth=None,focusing_axis=None):
    """Draw to scale, projecting the y-focusing element after rotation about z."""
    drawing=Schematic(418,242,ax)
    if show_depth is None:
        show_depth=cfg.grid_dimension==2 and cfg.geometry=='one-dimensional'
    depth=cfg.structure_depth_m if show_depth else None
    aperture,length,bridge=cfg.aperture_m,result.element_length_m,cfg.bridge_thickness_m
    diagonal=1/math.sqrt(2)
    perpendicular=focusing_axis=='y'
    # View the y stack from above, with its hidden face below the top face.
    focusing=np.array([diagonal,-diagonal] if perpendicular else [0.,1.])
    extrusion=np.array([0.,1.] if perpendicular else [diagonal,-diagonal])
    extent=aperture*abs(focusing)+(depth or 0)*abs(extrusion)
    scale=min(258/(length+extent[0]),126/extent[1])
    extrude=(depth or 0)*scale*extrusion
    center=np.array([203.,127.])-extrude/2
    half_length,half_aperture=length*scale/2,aperture*scale/2

    def project(z,u):
        return center+np.expand_dims(z,-1)*np.array([1.,0.])+np.expand_dims(u,-1)*focusing

    transverse=np.linspace(-aperture/2,aperture/2,65)
    longitudinal=(bridge/2+transverse**2/(2*cfg.radius_m))*scale
    outline=np.vstack((project(-longitudinal,transverse*scale),project(longitudinal,transverse*scale)[::-1]))
    if depth is not None:
        drawing.path(outline+extrude,'xp-layout-hidden xp-element-depth',close=True)
        for corner in (outline[0],outline[64],outline[65],outline[-1]):
            drawing.path([corner,corner+extrude],'xp-layout-hidden')
    drawing.path(outline,'xp-layout-lens xp-element-outline',close=True)

    def dimension(symbol,start,stop,offset,style,label_offset):
        if np.linalg.norm(np.subtract(stop,start))<21:
            return
        x,y=drawing.ruler(start,stop,offset,style)
        drawing.label(x+label_offset[0],y+label_offset[1],symbol,'xp-layout-small xp-dimension-symbol')

    end1,end2=project(-half_length,-half_aperture),project(-half_length,half_aperture)
    dimension('A',end1,end2,(-12,-12) if perpendicular else (-14,0),'xp-element-aperture',(-10,4))
    pitch_label=f'p = {analytical_number(length*1e6)} µm' if show_values else 'p'
    bottom_aperture=-half_aperture if perpendicular else half_aperture
    pitch_offset=extrude if perpendicular else (0.,0.)
    dimension(pitch_label,project(-half_length,bottom_aperture)+pitch_offset,
              project(half_length,bottom_aperture)+pitch_offset,
              (0,17),'xp-element-length',(0,15))
    d=bridge*scale/2
    center_x,center_y=center
    if 2*d<21:
        reference_y=max(54,min(end1[1],end2[1])-9)
        for sign in (-1,1):
            tip=center_x+sign*d
            line_style='xp-layout-dimension xp-element-bridge'
            drawing.path([(tip,center_y),(tip,reference_y-5)],line_style)
            drawing.path([(tip+sign*20,reference_y),(tip,reference_y)],line_style)
            drawing.path([(tip,reference_y),(tip+sign*7,reference_y-2.6),
                          (tip+sign*7,reference_y+2.6)],'xp-dimension-arrow xp-bridge-arrow',close=True)
        drawing.label(center_x+d+25,reference_y+4,'d','xp-layout-small xp-dimension-symbol','start')
    else:
        dimension('d',project(-d,0),project(d,0),(0,-13),'xp-element-bridge',(0,-6))
    if depth is not None:
        depth_start=project(half_length,half_aperture)
        dimension('L',depth_start,depth_start+extrude,(14,1) if perpendicular else (10,9),
                  'xp-element-depth-ruler',(9,4) if perpendicular else (13,9))

    vertex=center_x+d
    radius=cfg.radius_m*scale
    if 4<=radius and vertex+2*radius<=373:
        theta=np.linspace(0,math.tau,51)
        drawing.path(project(d+radius*(1+np.cos(theta)),radius*np.sin(theta)),'xp-radius-marker',close=True)
    elif radius>=4:
        u=np.linspace(-min(16,.85*radius),min(16,.85*radius),19)
        sag=u*u/(radius+np.sqrt(radius*radius-u*u))
        drawing.path(project(d+sag,u),'xp-radius-marker')
    arrow=min(24,max(6,radius))
    drawing.path([(vertex+arrow,center_y),(vertex,center_y),(vertex+4,center_y-2.4)],'xp-radius-marker')
    drawing.path([(vertex,center_y),(vertex+4,center_y+2.4)],'xp-radius-marker')
    drawing.label(vertex+arrow+5,center_y-5,'R','xp-layout-small xp-dimension-symbol','start')

    heading='Lens element cross-section' if focusing_axis is None else f'{focusing_axis}-focusing lens element'
    if show_values:
        drawing.label(209,20,heading)
        drawing.label(209,39,schematic_material(cfg),'xp-layout-small')
        if 2*half_length<21:drawing.label(209,235,pitch_label,'xp-layout-small')
    elif ax is not None:
        ax.set(xlim=(49,375),ylim=(228,50))
    css='xp-element-view'
    if focusing_axis is not None:
        css+=f' xp-element-focus-{focusing_axis}'
    return drawing.html(css,heading+' to scale')


def draw_line_focus_stack(drawing,project,entrance,exit_z,half_aperture,depth,count,web_fraction):
    slab_height=half_aperture+7
    pitch=(exit_z-entrance)/count
    cavity_half_length=pitch*(1-web_fraction)/2
    levels=np.linspace(-half_aperture,half_aperture,33)

    def path(points,style):
        drawing.path([project(*p) for p in points],style,close=True)

    def plate_face(y):
        return [(entrance,-slab_height,y),(exit_z,-slab_height,y),
                (exit_z,slab_height,y),(entrance,slab_height,y)]

    def cavity(z,y):
        left=[(z-cavity_half_length*(1-(x/half_aperture)**2),x,y) for x in levels]
        right=[(z+cavity_half_length*(1-(x/half_aperture)**2),x,y) for x in reversed(levels)]
        return left+right

    front=plate_face(depth)
    if depth:
        rear=plate_face(-depth)
        path(rear,'xp-layout-hidden')
        path([rear[3],rear[2],front[2],front[3]],'xp-layout-slab')
        path([rear[0],rear[3],front[3],front[0]],'xp-layout-slab')
        path([rear[1],rear[2],front[2],front[1]],'xp-layout-slab xp-layout-exit-face')

    stations=[entrance+(i+.5)*pitch for i in range(count)]
    holes=[cavity(z,depth) for z in stations]
    faces=[[project(*p) for p in surface] for surface in (front,*holes)]
    drawing.compound(faces,'xp-layout-slab')
    if drawing.ax is None:
        drawing.compound(faces,'xp-crl-hatch')
    for z,hole in zip(stations,holes):
        if depth:
            path(cavity(z,depth/4),'xp-layout-hidden')
            path(cavity(z,-depth/2),'xp-layout-hidden')
        path(hole,'xp-layout-cavity xp-layout-opening')


OPTICAL_SVG_FILLS=(
    '<defs><pattern id="xp-crl-hatch" width="5" height="5" patternUnits="userSpaceOnUse">'
    '<path class="xp-layout-hatch" d="M -1 1 L 1 -1 M 0 5 L 5 0 M 4 6 L 6 4"/></pattern>'
    '<radialGradient id="xp-gaussian" cx=".5" cy=".5" r=".5">'
    '<stop offset="0" stop-color="#ffdf7f"/><stop offset=".5" stop-color="#e1949f" stop-opacity=".8"/>'
    '<stop offset="1" stop-color="#a367c6" stop-opacity=".15"/></radialGradient></defs>')


def gaussian_source_radii(cfg,maximum_radius):
    widths=np.array([cfg.source_fwhm_x_m,cfg.source_fwhm_y_m])
    # Keep the source in the drawing area while preserving visible anisotropy.
    largest=max(float(widths.max()),5e-6)
    radius=schematic_extent(largest,250e-6,9,maximum_radius)
    return np.maximum(3.,radius*np.sqrt(widths/largest))


def optical_schematic(cfg,result,*,ax=None,show_values=True):
    """Draw the optical train in projection or isometry."""
    if cfg.geometry=='crossed':
        return crossed_schematic(cfg,result,ax=ax,show_values=show_values)
    iso=cfg.grid_dimension==2
    round_lens=cfg.geometry=='two-dimensional'
    real_image=result.real_image
    plane=cfg.source_type=='plane'
    extended=cfg.source_type=='gaussian'
    source_label={'point':'Point source','plane':'Plane wave','gaussian':'Gaussian source'}[cfg.source_type]
    lens_label='Point-focus CRL' if round_lens else 'Line-focus CRL'
    source_z=69 if plane else 37+64/(1+cfg.source_distance_m/15)
    count=min(cfg.n_elements,5 if round_lens else 4)
    half_length=schematic_extent(result.lens_length_m,.00265,31,115)
    entrance,exit_z=287-half_length,287+half_length
    pitch=(exit_z-entrance)/count
    distance=result.image_distance_m if real_image else .034
    focus=exit_z+(553-exit_z)*schematic_extent(distance,.034,.54,.94)
    half_aperture=schematic_extent(cfg.aperture_m,50e-6,15,38)
    slab_height=half_aperture+7
    depth=schematic_extent(cfg.structure_depth_m,30e-6,7,22) if iso and not round_lens else 0
    web_fraction=min(.6,max(.12,cfg.bridge_thickness_m/result.element_length_m))
    drawing=Schematic(600,236,ax)

    def project(z,x=0,y=0):
        if round_lens:
            # Preserve space above the cylindrical housing for its annotations.
            return z-.31*(x+y),141+.72*(y-x)
        if iso:
            return z-.5*(x+y),125+math.sqrt(3)/2*(y-x)
        return z,125-x

    def path(points,style,close=False):
        drawing.path([project(*p) for p in points],style,close=close)

    def line(a,b,style):
        path([a,b],style)

    label=drawing.label

    def dot(p,r,style):
        drawing.dot(*project(*p),r,style)

    cavity_half_length=pitch*(1-web_fraction)/2
    stations=[entrance+(i+.5)*pitch for i in range(count)]

    def draw_round_stack():
        # The viewing sector is open over the central cavities only. The
        # cross-sections at the window boundaries still carry the two end rims.
        opening=1.04
        casing_radius=slab_height
        visible=stations[1:-1] if count>=3 else stations
        margin=pitch/2-cavity_half_length
        window_start=visible[0]-pitch/2
        window_end=visible[-1]+pitch/2
        if count<3:
            window_start+=margin/2
            window_end-=margin/2

        def radial(z,radius,angle):
            theta=angle+5*np.pi/4
            return (z,radius*math.cos(theta),radius*math.sin(theta))

        def arc(z,radius,start,stop):
            return [radial(z,radius,t) for t in np.linspace(start,stop,25)]

        def skin(start,stop,angle_start,angle_stop,style):
            path(arc(start,casing_radius,angle_start,angle_stop)
                 +arc(stop,casing_radius,angle_stop,angle_start),style,True)

        path(arc(exit_z,casing_radius,0,math.tau),'xp-round-cap',True)
        skin(entrance,exit_z,np.pi/2,3*np.pi/2,'xp-round-back')
        for z in (window_start,window_end):
            sector=[(z,0,0),*arc(z,casing_radius,-opening,opening)]
            path(sector,'xp-round-cut xp-round-window-rim',True)

        angles=np.linspace(opening,math.tau-opening,25)
        meridians=(1.43,2.55,3.68,4.78)
        for z in reversed(visible):
            for sign,style in ((-1,'xp-round-cavity-far'),(1,'xp-round-cavity-near')):
                def surface(fraction,angle):
                    axial=z+sign*cavity_half_length*(1-fraction**2)
                    return radial(axial,half_aperture*fraction,angle)
                rim=[surface(1,t) for t in angles]
                shoulder=[surface(.66,t) for t in angles]
                path(rim+shoulder[::-1],style,True)
                path(shoulder+[surface(0,0)],style,True)
                path(shoulder,'xp-round-mesh')
                for angle in meridians:
                    path([surface(fraction,angle) for fraction in np.linspace(0,1,15)],'xp-round-mesh')

        # Two radial cut planes bound the window. Each strip of material
        # follows the parabolas of its adjacent cavities on the same scale.
        radius=np.r_[np.linspace(0,half_aperture,26),casing_radius]
        sag=cavity_half_length*np.maximum(0,1-(radius/half_aperture)**2)
        left_edges=[np.full_like(radius,window_start),*(z+sag for z in visible)]
        right_edges=[*(z-sag for z in visible),np.full_like(radius,window_end)]
        for angle in (-opening,opening):
            for left_edge,right_edge in zip(left_edges,right_edges):
                near=[radial(z,r,angle) for z,r in zip(left_edge,radius)]
                far=[radial(z,r,angle) for z,r in zip(right_edge[::-1],radius[::-1])]
                path(near+far,'xp-round-cut',True)
            line(radial(window_start,casing_radius,angle),radial(window_end,casing_radius,angle),'xp-round-window-edge')

        bands=(
            (-np.pi/2,-opening,'xp-round-case-top'),
            (-opening,-.40,'xp-round-case-light'),
            (-.40,.40,'xp-round-case-mid'),
            (.40,opening,'xp-round-case-bottom'),
            (opening,np.pi/2,'xp-round-case-bottom'),
        )
        for a,b,style in bands:
            skin(entrance,window_start,a,b,style)
            skin(window_end,exit_z,a,b,style)
        skin(window_start,window_end,-np.pi/2,-opening,'xp-round-case-top')
        skin(window_start,window_end,opening,np.pi/2,'xp-round-case-bottom')
        for angle in (-np.pi/2,np.pi/2):
            line(radial(entrance,casing_radius,angle),radial(exit_z,casing_radius,angle),'xp-round-window-edge')
        path(arc(entrance,casing_radius,0,math.tau),'xp-round-cap',True)
        path(arc(entrance,half_aperture,0,math.tau),'xp-round-aperture',True)

    line((28,0,0),(557,0,0),'xp-layout-axis')
    label(571,145 if round_lens else 129,'z','xp-layout-small')

    if extended:
        if iso:
            size_x,size_y=gaussian_source_radii(cfg,28)
            source_face=[(source_z,size_x*math.cos(t*math.tau/48),size_y*math.sin(t*math.tau/48)) for t in range(49)]
            path(source_face,'xp-layout-source-shape',True)
        else:
            sigma=schematic_extent(cfg.source_fwhm_x_m,5e-6,3,15)
            xs=[sigma*i/6 for i in range(-18,19)]
            source_face=[(source_z+16*math.exp(-.5*(x/sigma)**2),x,0) for x in xs]
            path([(source_z,xs[0],0),*source_face,(source_z,xs[-1],0)],'xp-layout-source-shape',True)
        if cfg.source_method=='points':
            width= size_x if iso else 2*sigma
            for offset in (-.85,-.42,0,.42,.85):
                dot((source_z,width*offset,0),1.8,'xp-layout-source')
    elif not plane:
        dot((source_z,0,0),5,'xp-layout-source')

    if round_lens:
        draw_round_stack()
    else:
        draw_line_focus_stack(drawing,project,entrance,exit_z,half_aperture,depth,count,web_fraction)

    ray_x=.54*half_aperture if round_lens else .76*half_aperture
    ray_y=ray_x if round_lens else .72*depth
    ray_points=[(x,y) for x in (-ray_x,ray_x) for y in ((-ray_y,ray_y) if iso else (0,))]
    if round_lens:
        ray_points=[(-ray_x,ray_y),(ray_x,-ray_y)]
    for x,y in ray_points:
        start=(30,x,y) if plane else (source_z,x*.55 if extended else 0,y*.5 if extended else 0)
        incident_edge=(entrance,x,y)
        exit_y=y*.88 if round_lens else y
        exit_edge=(exit_z,x*.82,exit_y)
        if real_image:
            final_y=0 if round_lens else y
            downstream=[(focus,0,final_y),(551,-x*.20,final_y)]
        else:
            downstream=[(535,x*1.32,exit_y)]
        if round_lens:
            line(start,incident_edge,'xp-layout-ray')
            line(incident_edge,exit_edge,'xp-round-internal-ray')
            path([exit_edge,*downstream],'xp-layout-ray')
        else:
            path([start,incident_edge,exit_edge,*downstream],'xp-layout-ray')

    if show_values:
        label(12,30,source_label,anchor='start')
        label(12,49,analytical_number(cfg.energy_kev)+' keV','xp-layout-small','start')
        lens_label_z=(entrance+exit_z)/2+20
        label(lens_label_z,30,lens_label)
        label(lens_label_z,47,f'{cfg.n_elements} elements','xp-layout-small')
        label(lens_label_z,63,schematic_material(cfg),'xp-layout-small')
    if real_image:
        if plane:
            focus_label='Point focus' if round_lens else 'Line focus'
        else:
            focus_label='Image point' if round_lens else 'Line image'
        if iso and not round_lens:
            line((focus,0,-23),(focus,0,23),'xp-layout-focus-line')
        else:
            dot((focus,0,0),5,'xp-layout-focus')
        if show_values:label(min(focus,517),30,focus_label)
        drawing.ruler((exit_z,184),(focus,184),(0,6))
        if show_values:
            caption_x=min((exit_z+focus)/2,463)
            distance_label='Back focal distance' if plane else 'Image distance (analytical)'
            label(caption_x,209,distance_label,'xp-layout-small')
            label(caption_x,226,html.unescape(analytical_length(result.image_distance_m)),'xp-layout-small')
    else:
        image_label=('Image at infinity' if math.isinf(result.image_distance_m or 0) else 'Virtual image')
        if show_values:label(587,30,image_label if result.external_focus else 'Focus inside CRL',anchor='end')
    if not plane:
        drawing.ruler((source_z,184),(entrance,184),(0,6))
        if show_values:
            label((source_z+entrance)/2,207,analytical_number(cfg.source_distance_m)+' m to CRL entrance','xp-layout-small')
    if extended and show_values:
        widths='x '+analytical_number(cfg.source_fwhm_x_m*1e6)
        if iso:
            widths+=' · y '+analytical_number(cfg.source_fwhm_y_m*1e6)
        label(12,66,'FWHM '+widths+' µm','xp-layout-small','start')
    line((546,-36,0),(546,-19,0),'xp-layout-axis')
    label(*project(546,-15,0),'x','xp-layout-small')
    if iso:
        line((546,-36,0),(546,-36,22),'xp-layout-axis')
        px,py=project(546,-36,29)
        label(px,py+4,'y','xp-layout-small')

    if not show_values and ax is not None:
        label((entrance+exit_z)/2,73,f'N = {cfg.n_elements}','xp-layout-small')
        ax.set(xlim=(21,590),ylim=(202,52))
    description='Optical layout with '+source_label+' and '+lens_label+(' in isometric view' if iso else ' in x-z projection')
    return drawing.html('xp-optical-layout',description,OPTICAL_SVG_FILLS)


def alignment_length(value):
    if value is None or not math.isfinite(value):
        return analytical_number(value)
    scale,unit=(1e-3,'mm') if abs(value)>=1e-4 else (1e-6,'µm')
    return decimal_number(value/scale)+' '+unit


def crossed_schematic(cfg,analysis,*,ax=None,show_values=True):
    """Plot the x stack and the rotated y stack on the same optical axis."""
    drawing=Schematic(650,236,ax)
    gap=analysis.gap_m
    lx,ly=analysis.x_lens.lens_length_m,analysis.y_lens.lens_length_m
    positions=(analysis.common_image_m,)*2 if analysis.aligned else (analysis.image_x_m,analysis.image_y_m)
    farthest=max((z for z in positions if z is not None and math.isfinite(z) and z>0),default=.03)
    upstream=147
    widths=[min(88,max(53,86*math.sqrt(length/max(lx,ly)))) for length in (lx,ly)]
    exit_x=upstream+widths[0]
    entrance_y=exit_x+(62 if gap is None else min(102,58+13*math.log1p(gap/max(lx,ly))))
    exit_y=entrance_y+widths[1]
    y_lens=cfg.second_crl
    source_z=52
    centers=((upstream+exit_x)/2,(entrance_y+exit_y)/2)

    def project(z,x=0,y=0):
        return z-.5*(x+y),122+math.sqrt(3)/2*(y-x)

    def project_y(z,u,v):
        return project(z,-v,u)

    def line(points,style):
        drawing.path([project(*p) for p in points],style)

    slits=((cfg,analysis.x_lens,project,upstream,exit_x,1),
           (y_lens,analysis.y_lens,project_y,entrance_y,exit_y,-1))
    for lens,analytic,projection,start,stop,side in slits:
        aperture=schematic_extent(lens.aperture_m,50e-6,15,30)
        depth=side*schematic_extent(lens.structure_depth_m,30e-6,7,19)
        web=min(.6,max(.12,lens.bridge_thickness_m/analytic.element_length_m))
        draw_line_focus_stack(drawing,projection,start,stop,aperture,depth,min(lens.n_elements,4),web)

    def image_mark(z):
        if z is None or not math.isfinite(z) or z<=0:
            return None
        return exit_y+23+(578-exit_y-23)*z/farthest

    line([(29,0,0),(629,0,0)],'xp-layout-axis')
    amplitudes=[schematic_extent(lens.aperture_m,50e-6,9,17)*.74 for lens in (cfg,y_lens)]
    for axis in (0,1):
        focus=image_mark(positions[axis])
        center=centers[axis]
        style=('xp-crossed-x','xp-crossed-y')[axis]
        for sign in (-1,1):
            amplitude=sign*amplitudes[axis]
            start=amplitude if cfg.source_type=='plane' else .25*amplitude if cfg.source_type=='gaussian' else 0.
            def ray(z,t):
                return (z,t,0) if axis==0 else (z,0,t)
            endpoints=[ray(source_z,start),ray(center,amplitude)]
            if focus is None:
                endpoints.append(ray(613,1.12*amplitude))
            else:
                endpoints.extend((ray(focus,0),ray(613,-amplitude*min(.65,(613-focus)/(focus-center)))))
            line(endpoints,style)
        if focus is not None and not analysis.aligned:
            if axis==0:
                line([(focus,0,-12),(focus,0,12)],style+' xp-crossed-plane')
            else:
                line([(focus,-12,0),(focus,12,0)],style+' xp-crossed-plane')
            if show_values:
                drawing.label(focus,100 if axis==0 else 149,('x image','y image')[axis],
                              'xp-layout-small')
    if cfg.source_type=='point':
        drawing.dot(*project(source_z),4,'xp-layout-source')
    elif cfg.source_type=='gaussian':
        sx,sy=gaussian_source_radii(cfg,21)
        source_outline=[project(source_z,sx*np.cos(t),sy*np.sin(t)) for t in np.linspace(0,math.tau,48)]
        drawing.path(source_outline,'xp-layout-source-shape',close=True)

    common=image_mark(analysis.common_image_m)
    if common is not None:
        drawing.dot(*project(common),4.6,'xp-layout-focus')
    if show_values:
        names={'point':'Point source','gaussian':'Gaussian source','plane':'Plane wave'}
        drawing.label(14,34,names[cfg.source_type],'xp-layout-small xp-crossed-source','start')
        drawing.label(14,53,analytical_number(cfg.energy_kev)+' keV','xp-layout-small','start')
        if cfg.source_type=='gaussian':
            for axis,width,y in (('x',cfg.source_fwhm_x_m,70),('y',cfg.source_fwhm_y_m,87)):
                drawing.label(14,y,f'{axis}: {analytical_number(width*1e6)} µm','xp-layout-small','start')
        for axis,lens,center in zip(('x','y'),(cfg,y_lens),centers):
            drawing.label(center,32,axis+' CRL')
            drawing.label(center,50,f'N = {lens.n_elements}', 'xp-layout-small xp-stack-count')
            drawing.label(center,67,schematic_material(lens),'xp-layout-small xp-crossed-secondary')
        drawing.label(632,34,'Common image plane' if analysis.aligned else 'No common downstream image',
                      'xp-layout-label','end')
        if cfg.source_distance_m is not None:
            drawing.ruler((source_z,173),(upstream,173),(0,10))
            drawing.label((source_z+upstream)/2,205,analytical_number(cfg.source_distance_m)+' m','xp-layout-small')
        drawing.ruler((exit_x,173),(entrance_y,173),(0,10))
        drawing.label((exit_x+entrance_y)/2,204,alignment_length(gap),'xp-layout-small xp-crossed-secondary')
        drawing.label((exit_x+entrance_y)/2,222,'Gap','xp-layout-small xp-crossed-secondary')
        if common is not None:
            drawing.ruler((exit_y,173),(common,173),(0,10))
            drawing.label((exit_y+common)/2,204,alignment_length(analysis.common_image_m),'xp-layout-small xp-crossed-secondary')
            drawing.label((exit_y+common)/2,222,'Image distance from y exit','xp-layout-small xp-crossed-secondary')
        origin=(606,-21,15)
        line([origin,(626,-21,15)],'xp-layout-axis')
        line([origin,(606,-3,15)],'xp-layout-axis')
        line([origin,(606,-21,32)],'xp-layout-axis')
        for label,point in (('x',(605,1,15)),('y',(603,-21,36)),('z',(635,-21,15))):
            drawing.label(*project(*point),label,'xp-layout-small')
    elif ax is not None:
        drawing.label(45,91,'Source','xp-layout-small')
        for axis,lens,center in zip(('x','y'),(cfg,y_lens),centers):
            drawing.label(center,77,f'{axis} CRL · N = {lens.n_elements}','xp-layout-small')
        ax.set(xlim=(25,632),ylim=(172,74))

    diagram=drawing.html('xp-optical-layout xp-crossed-layout','Isometric optical layout with an x CRL and a y CRL. '+analysis.message,
                         OPTICAL_SVG_FILLS)
    if not show_values:
        return diagram
    note=analysis.message
    precise_gap=gap
    if not analysis.aligned and analysis.recommended_gap_m is not None:
        precise_gap=analysis.recommended_gap_m
        note+=' Gap for a common image plane: '+alignment_length(precise_gap)+'.'
    precise_label='' if precise_gap is None else f' title="Gap at higher precision: {decimal_number(precise_gap*1e6,3)} µm"'
    mobile=[('Gap',alignment_length(gap))]
    if common is not None:
        mobile.append(('Image distance',alignment_length(analysis.common_image_m)))
    readouts='<dl class="xp-crossed-mobile">'+''.join(
        f'<div><dt>{k}</dt><dd>{v}</dd></div>' for k,v in mobile)+'</dl>'
    return diagram+readouts+('<div class="xp-crossed-status" data-aligned="'+str(analysis.aligned).lower()+'"'
                             +precise_label+'>'+html.escape(note)+'</div>')


def material_readout(result):
    return '<dl class="xposim-analytical-list">'+''.join((
        analytical_entry('δ · refractive index decrement',analytical_number(result.delta,True)),
        analytical_entry('β · absorption index',analytical_number(result.beta,True)),
        analytical_entry('Web transmission',analytical_number(100*result.web_transmission)+' %'),
        analytical_entry('Rocking curve FWHM (analytical)','Unresolved' if result.rocking_fwhm_rad is None else
                         analytical_number(1e3*result.rocking_fwhm_rad)+' mrad'),
    ))+'</dl>'


def show_crossed_preview(cfg,result):
    optical_layout.value=crossed_schematic(cfg,result)
    element_preview.value=element_schematic(cfg,result.x_lens,show_depth=True,focusing_axis='x')
    second_element_preview.value=element_schematic(cfg.second_crl,result.y_lens,show_depth=True,focusing_axis='y')
    preflight_material.value=material_readout(result.x_lens)
    second_preflight_material.value=material_readout(result.y_lens)
    separation=result.image_separation_m
    analytic=[
        ('Inter-stack gap · x exit to y entrance',alignment_length(result.gap_m)),
        ('x image · from y exit',alignment_length(result.image_x_m)),
        ('y image · from y exit',alignment_length(result.image_y_m)),
        ('Image-plane separation',alignment_length(None if separation is None else abs(separation))),
    ]
    if result.gap_m is not None and cfg.source_type!='plane':
        rows=''.join(magnification_row(axis,value) for axis,value in zip(('x','y'),result.magnification_at_image))
        analytic.append(('Magnification at the axis images',rows))
    analytic.extend((
        ('x CRL length / maximum N',f'{alignment_length(result.x_lens.lens_length_m)} / {result.x_lens.max_element_count}'),
        ('y CRL length / maximum N',f'{alignment_length(result.y_lens.lens_length_m)} / {result.y_lens.max_element_count}'),
    ))
    preflight_geometry.value='<dl class="xposim-analytical-list">'+''.join(analytical_entry(k,v,css_class='xposim-magnification' if k=='Magnification at the axis images' else '') for k,v in analytic)+'</dl>'
    preflight_status.value='<div class="xp-layout-caption">The z origin is the y CRL exit. Lens lengths and the gap are included in both image coordinates.</div>'


def update_roi_status():
    values={name:widget.value for name,widget in controls.items()}
    color='var(--xp-secondary)'
    try:
        z,_=detector_bounds(values)
        exit_name='y CRL exit' if values['focusing']=='crossed' else 'CRL exit'
        reference=f'z is measured from the {exit_name}.'
        note=(reference+' x and y are measured from the optical axis.' if z is not None else
              'The beam determines the axial interval and transverse region automatically. '+reference)
    except ValueError as exc:
        note=str(exc);color='#b77d38'
    roi_status.value='<div class="xp-layout-caption" style="color:'+color+'">'+html.escape(note)+'</div>'


def update_analytical_preview():
    global _preflight_result
    try:
        cfg=read_configuration(include_detector=False);result=xpos.analyze_crl(cfg);_preflight_result=result
        if cfg.geometry=='crossed':
            show_crossed_preview(cfg,result)
            return
        optical_layout.value=optical_schematic(cfg,result)
        element_preview.value=element_schematic(cfg,result)
        web=analytical_number(100*result.web_transmission)+' %'
        rocking='Unresolved' if result.rocking_fwhm_rad is None else analytical_number(1e3*result.rocking_fwhm_rad)+' mrad'
        focal=analytical_length(result.collimated_focus_m);image=analytical_length(result.image_distance_m)
        maximum=str(result.max_element_count);length=analytical_length(result.lens_length_m)
        delta=analytical_number(result.delta,True);beta=analytical_number(result.beta,True)
        scales=result.magnification_at_image[:2 if cfg.geometry=='two-dimensional' else 1]
        if not any(value is not None for value in scales):
            magnification='—'
        elif cfg.geometry=='two-dimensional' and all(value is not None for value in scales) and math.isclose(*scales,rel_tol=1e-9):
            magnification=magnification_row('x, y',scales[0])
        else:
            labels=('x','y')
            magnification=''.join(magnification_row(labels[i],value) for i,value in enumerate(scales))
        if not result.external_focus:message=f'A focus beyond the CRL exit requires N ≤ {result.max_element_count}.'
        elif not result.real_image:message='The ray model predicts a virtual image or an image at infinity.'
        else:message='Back focal and image distances are measured from the CRL exit.'
        color='var(--xp-secondary)' if result.real_image else '#b77d38'
    except Exception as exc:
        _preflight_result=None
        optical_layout.value='<div class="xp-layout-caption">Enter valid source and CRL parameters to preview the layout.</div>'
        element_preview.value=''
        second_element_preview.value=''
        second_preflight_material.value=''
        focal=image=maximum=length=delta=beta=magnification=web=rocking='—';message=str(exc);color='#b77d38'
    grid='<dl class="xposim-analytical-list">'
    coherence=''
    if _preflight_result is not None and result.real_image:
        values=result.transverse_coherence_length_m(result.image_distance_m)
        if values is not None:
            coherence=analytical_entry('Transverse coherence length at image',coherence_values(values,analytical_length))
    geometry_entries=[analytical_entry('Back focal distance',focal)]
    if controls['source'].value!='plane':
        geometry_entries.extend([analytical_entry('Image distance (analytical)',image),
                                 analytical_entry('Transverse magnification',magnification,css_class='xposim-magnification')])
    geometry_entries.extend([analytical_entry('Maximum element count',maximum),analytical_entry('CRL length',length)])
    preflight_geometry.value=grid+''.join(geometry_entries)+coherence+'</dl>'
    preflight_material.value=(grid+analytical_entry('δ · refractive index decrement',delta)+analytical_entry('β · absorption index',beta)+
        analytical_entry('Web transmission',web)+analytical_entry('Rocking curve FWHM (analytical)',rocking)+'</dl>')
    preflight_status.value='<div style="font-size:12px;line-height:1.3;color:'+color+';padding-top:4px">'+html.escape(message)+'</div>'


def inputs_changed(change=None):
    if _syncing_controls:return
    sync_controls(change)
    update_analytical_preview()
    update_roi_status()
    if 'sync_results_notice' in globals():sync_results_notice()
for widget in controls.values():widget.observe(inputs_changed,names='value')
layout_section=section('Optical layout',[optical_layout],'optical-layout')
analytical_section=section('Analytical estimates',[preflight_geometry,preflight_status],'analytical')
overview=widgets.GridBox([layout_section,analytical_section],layout=widgets.Layout(width='100%'))
overview.add_class('xp-parameter-overview')
source_section=section('Source',[source_controls],'source')
geometry_inputs=widgets.VBox([general_controls,geometry_controls],layout=widgets.Layout(width='100%',gap='7px'))
geometry_layout=widgets.GridBox([geometry_inputs,element_preview],layout=widgets.Layout(width='100%'))
geometry_layout.add_class('xp-geometry-fields')
geometry_section=section('CRL geometry',[geometry_layout],'geometry')

material_layout=widgets.GridBox([material_controls,preflight_material],layout=widgets.Layout(width='100%'))
material_layout.add_class('xp-material-fields')
material_section=section('Material',[material_layout],'material')
second_geometry_layout=widgets.GridBox([second_geometry_controls,second_element_preview],layout=widgets.Layout(width='100%'))
second_geometry_layout.add_class('xp-geometry-fields')
second_material_layout=widgets.GridBox([second_material_controls,second_preflight_material],layout=widgets.Layout(width='100%'))
second_material_layout.add_class('xp-material-fields')
second_stack_section=section('y-focusing CRL',[second_geometry_layout,
    widgets.HTML('<b>y-focusing CRL material</b>'),second_material_layout],'second-stack')
spacing_section=section('Crossed CRL alignment',[spacing_controls],'crossed-spacing')

roi_section=section('Calculation region',[roi_fields,roi_status,
    widgets.HTML('<b>Sampling and peak search</b>'),advanced_fields],'detector-region')
detector_toggle=widgets.Checkbox(value=_show_detector,description='Show scan region and sampling settings',indent=False,
    layout=widgets.Layout(width='100%',height='auto'))
detector_toggle.add_class('xp-detector-toggle')

def show_detector_settings(change):
    roi_section.layout.display='' if change['new'] else 'none'

detector_toggle.observe(show_detector_settings,names='value')
show_detector_settings({'new':detector_toggle.value})
for panel in (source_controls,general_controls,geometry_controls,material_controls,second_geometry_controls,
              second_material_controls,spacing_controls,roi_fields,advanced_fields):
    panel.add_class('xp-control-grid')
source_controls.add_class('xp-source-grid')
sync_controls();update_analytical_preview();update_roi_status()

UI_CSS='<style>\n.xposim-parameters,.xposim-explorer {\n --xp-bg:var(--colab-primary-surface-color,light-dark(#f3f6fa,#181d25));\n --xp-surface:var(--colab-secondary-surface-color,light-dark(#ffffff,#222a35));\n --xp-fg:var(--colab-primary-text-color,light-dark(#172235,#f1f5fa));\n --xp-secondary:var(--colab-secondary-text-color,light-dark(#465b74,#c0ccdd));\n --xp-border:light-dark(#9aaabe,#64768f);--xp-grid:light-dark(#d6dee9,#3c495c);\n --xp-heading:light-dark(#e6edf5,#303c4f);--xp-input:light-dark(#fff,#293445);\n box-sizing:border-box;color:var(--xp-fg);background:var(--xp-bg);color-scheme:light dark;padding:10px;\n}\n.xposim-parameters.xposim-dark,.xposim-explorer.xposim-dark {\n --xp-bg:#181d25;--xp-surface:#222a35;--xp-fg:#f1f5fa;--xp-secondary:#c0ccdd;\n --xp-border:#64768f;--xp-grid:#3c495c;--xp-heading:#303c4f;--xp-input:#293445;color-scheme:dark;\n}\n.xposim-parameters.xposim-light,.xposim-explorer.xposim-light {\n --xp-bg:#f3f6fa;--xp-surface:#fff;--xp-fg:#172235;--xp-secondary:#465b74;\n --xp-border:#9aaabe;--xp-grid:#d6dee9;--xp-heading:#e6edf5;--xp-input:#fff;color-scheme:light;\n}\n.xposim-parameters .widget-box,.xposim-explorer .widget-box,\n.xposim-parameters .widget-inline-hbox,.xposim-explorer .widget-inline-hbox,\n.xposim-parameters .widget-html,.xposim-explorer .widget-html {\n box-sizing:border-box;min-width:0;max-width:100%;margin:0;overflow:visible;\n}\n.xposim-parameters .widget-html,.xposim-explorer .widget-html {color:var(--xp-fg)}\n.xposim-parameters .widget-html-content *,.xposim-explorer .widget-html-content * {color:inherit}\n.xposim-parameters .widget-html-content,.xposim-explorer .widget-html-content {\n min-width:0;max-width:100%;overflow-wrap:normal;word-break:normal;color:inherit;\n}\n.xposim-parameters input,.xposim-parameters select,.xposim-explorer input,.xposim-explorer select {\n min-width:0;box-sizing:border-box;background:var(--xp-input);color:var(--xp-fg);border-color:var(--xp-border);\n}\n.xposim-parameters input:disabled,.xposim-parameters select:disabled {opacity:.58}\n.xposim-parameters .widget-label,.xposim-explorer .widget-label {color:var(--xp-fg)}\n.xposim-explorer .widget-image {display:block;min-width:0;max-width:100%;margin:0;height:auto}\n.xposim-parameters .xposim-section {border:1px solid var(--xp-border)!important;border-radius:4px;background:var(--xp-surface)}\n.xposim-analytical-list {margin:10px 0 0;padding:0;font-size:14px;line-height:1.4}\n.xposim-analytical-list>div {display:grid;grid-template-columns:minmax(0,1fr) auto;align-items:baseline;gap:6px 12px;padding:4px 0;border-bottom:1px solid var(--xp-grid)}\n.xposim-analytical-list>div:last-child {border-bottom:0}\n.xposim-parameters .xposim-analytical-list dt,.xposim-parameters .xposim-analytical-list dd {width:auto;float:none;clear:none;min-width:0;padding:0;margin:0}\n.xposim-parameters .xposim-analytical-list dt {font-weight:400;color:var(--xp-secondary)}\n.xposim-parameters .xposim-analytical-list dd {font-size:19px;font-weight:600;white-space:nowrap;font-variant-numeric:tabular-nums}\n.xposim-analytic-status,.xposim-material-note {color:var(--xp-secondary)}\n.xposim-explorer .xposim-run {border:1px solid var(--xp-border);margin:7px 0 8px;background:var(--xp-surface)}\n.xposim-explorer .xposim-group-title {font-size:14px;font-weight:600;padding:5px 10px;background:var(--xp-heading);border-bottom:1px solid var(--xp-border)}\n.xposim-explorer .xposim-run-table {width:100%;border-collapse:collapse;table-layout:fixed;font-variant-numeric:tabular-nums}\n.xposim-explorer .xposim-run-table th {font-size:14px;font-weight:500;text-align:left;vertical-align:bottom;padding:6px 10px;color:var(--xp-fg)}\n.xposim-explorer .xposim-run-table td {font-size:21px;font-weight:600;text-align:left;padding:1px 10px 7px}\n.xposim-explorer .xposim-run-table th+th,.xposim-explorer .xposim-run-table td+td {border-left:1px solid var(--xp-border)}\n.xposim-explorer .xposim-unit {display:block;font-size:12px;font-weight:400;color:var(--xp-secondary)}\n.xposim-explorer .xposim-panel {border:1px solid var(--xp-border);background:var(--xp-surface);min-width:0}\n.xposim-explorer .xposim-panel-heading {display:flex;gap:10px;align-items:baseline;justify-content:space-between;flex-wrap:wrap;font-size:15px;font-weight:600;padding:7px 10px;background:var(--xp-heading);border-bottom:1px solid var(--xp-border)}\n.xposim-explorer .xposim-selected-z {font-size:20px;font-weight:600;font-variant-numeric:tabular-nums}\n.xposim-explorer .xposim-plane-metrics {display:grid;grid-template-columns:minmax(0,.9fr) minmax(0,1.1fr);margin:0 9px 8px;gap:9px}\n.xposim-explorer .xposim-metric-box {border:1px solid var(--xp-border);background:var(--xp-surface);min-width:0}\n.xposim-explorer .xposim-metric-title {font-size:14px;font-weight:500;padding:4px 8px;background:var(--xp-heading);border-bottom:1px solid var(--xp-border)}\n.xposim-explorer .xposim-metric-value {font-size:21px;font-weight:600;padding:4px 8px;font-variant-numeric:tabular-nums}\n.xposim-explorer .xposim-axis-table {width:100%;border-collapse:collapse;table-layout:fixed;font-size:14px;font-variant-numeric:tabular-nums}\n.xposim-explorer .xposim-axis-table th {font-size:12px;font-weight:400;text-align:right;padding:2px 8px;color:var(--xp-secondary)}\n.xposim-explorer .xposim-axis-table td {font-size:19px;font-weight:600;text-align:right;padding:2px 8px}\n.xposim-explorer .xposim-axis-table th:first-child,.xposim-explorer .xposim-axis-table td:first-child {text-align:left;width:43px}\n.xposim-explorer .xposim-axis-table td:first-child {font-size:13px;font-weight:400}\n.xposim-explorer .xposim-axis-table tr+tr {border-top:1px solid var(--xp-grid)}\n.xposim-axis-badge {display:inline-flex;align-items:center;gap:5px;white-space:nowrap}\n.xposim-axis-badge>span {display:inline-block;width:12px}\n.xposim-explorer .xposim-selection-controls {padding:6px 8px;margin:8px 0 5px;border:1px solid var(--xp-border);background:var(--xp-heading)}\n.xposim-explorer .jupyter-button {border:1px solid var(--xp-border);border-radius:4px;background:var(--xp-input);color:var(--xp-fg)}\n.xposim-explorer .jupyter-button:focus-visible {outline:2px solid #38c5b6;outline-offset:2px}\n.xposim-explorer .widget-button.mod-primary {background:#355f85;color:#fff}\n.xposim-explorer .noUi-target {background:var(--xp-border)}\n.xposim-explorer .xposim-notice {font-size:13px;padding:5px 8px;border-left:3px solid var(--xp-border);background:var(--xp-heading)}\n.xposim-explorer .xposim-runtime {color:var(--xp-secondary)}\n.xposim-explorer .xposim-runtime b {color:var(--xp-fg)}\n.xposim-parameters .widget-html-content,.xposim-parameters .widget-html-content *,\n.xposim-explorer .widget-html-content,.xposim-explorer .widget-html-content * {line-height:1.3}\n.xposim-explorer .xposim-run {margin:4px 0 5px}\n.xposim-explorer .xposim-group-title {padding:3px 10px;font-size:13px}\n.xposim-explorer .xposim-run-table th .xposim-unit {display:inline;margin-left:5px}\n.xposim-explorer .xposim-metric-title .xposim-unit {display:inline;margin-left:5px}\n.xposim-explorer .xposim-axis-table td {padding-top:0;padding-bottom:0}\n\n.xposim-parameters,.xposim-explorer,\n.xposim-parameters .widget-html-content,.xposim-explorer .widget-html-content,\n.xposim-parameters .widget-label,.xposim-explorer .widget-label,\n.xposim-parameters input,.xposim-parameters select,.xposim-explorer input,.xposim-explorer select,\n.xposim-explorer .jupyter-button {\n font-family:Consolas,\'DejaVu Sans Mono\',\'Liberation Mono\',monospace;\n}\n.xposim-parameters .widget-html-content *,.xposim-explorer .widget-html-content * {font-family:inherit}\n.xposim-explorer .xposim-single-metric {grid-template-columns:1fr}\n\n/* The output width can be smaller than the viewport (Colab side panels/iframes). */\n.xposim-parameters {container:xp-parameters / inline-size}\n.xposim-explorer {container:xp-explorer / inline-size}\n.xposim-parameters .xposim-section {container:xp-section / inline-size}\n.xposim-parameters .xposim-field-caption {height:auto;min-height:27px}\n.xposim-parameters .xposim-input-field>input,\n.xposim-parameters .xposim-input-field>select {flex-shrink:0}\n.xposim-parameters .xposim-analytical-list dd {white-space:normal}\n.xposim-parameters .xposim-number {white-space:nowrap;display:inline-block}\n.xposim-parameters .xposim-analytical-list .xposim-magnification {grid-template-columns:1fr}\n.xposim-magnification-axis {display:flex;flex-wrap:wrap;align-items:baseline;gap:2px 12px}\n.xposim-magnification-axis+.xposim-magnification-axis {margin-top:4px}\n.xposim-magnification-axis .xposim-axis-name {font-size:13px;font-weight:400;white-space:nowrap}\n.xposim-explorer .xposim-run-metrics {display:grid;grid-template-columns:repeat(var(--xp-metric-count),minmax(0,1fr));margin:0;padding:0}\n.xposim-explorer .xposim-run-metrics>div {display:flex;flex-direction:column;min-width:0;padding:7px 10px;border-right:1px solid var(--xp-border)}\n.xposim-explorer .xposim-run-metrics>div:last-child {border-right:0}\n.xposim-explorer .xposim-run-metrics dt {flex:1;float:none;width:auto;margin:0;padding:0 0 5px;font-size:14px;line-height:1.4;font-weight:500}\n.xposim-explorer .xposim-run-metrics dd {float:none;width:auto;margin:0;padding:0;font-size:21px;line-height:1.3;font-weight:600;white-space:nowrap;font-variant-numeric:tabular-nums}\n.xposim-explorer .xposim-run-metrics .xposim-unit {display:inline;margin-left:6px;white-space:nowrap}\n.xposim-explorer .xposim-toolbar {display:grid;grid-template-columns:125px minmax(0,1fr) 160px;grid-template-areas:"compute runtime theme"}\n.xposim-toolbar>.widget-button {grid-area:compute}\n.xposim-toolbar>.xposim-runtime {grid-area:runtime}\n.xposim-toolbar>.xposim-theme-selector {grid-area:theme}\n.xposim-explorer .xposim-selected-z,.xposim-explorer .xposim-plane-label {white-space:nowrap}\n.xposim-explorer .xposim-selection-row {gap:4px}\n@container xp-section (max-width:460px) {\n .xposim-analytical-list>div {grid-template-columns:1fr;gap:3px}\n .xposim-parameters .xposim-analytical-list dd {font-size:17px}\n}\n@container xp-parameters (max-width:400px) {\n .xposim-parameters .xposim-section {padding:10px 8px!important}\n}\n@container xp-explorer (max-width:900px) {\n .xposim-explorer .xposim-run-metrics {grid-template-columns:repeat(2,minmax(0,1fr))}\n .xposim-explorer .xposim-run-metrics>div {border-right:0;border-bottom:1px solid var(--xp-border)}\n .xposim-explorer .xposim-run-metrics>div:nth-child(odd) {border-right:1px solid var(--xp-border)}\n .xposim-explorer .xposim-run-metrics>div:last-child:nth-child(odd) {grid-column:1/-1;border-right:0}\n}\n@container xp-explorer (max-width:700px) {\n .xposim-explorer .xposim-toolbar {grid-template-columns:minmax(90px,1fr) minmax(100px,1fr);grid-template-areas:"compute theme" "runtime runtime";gap:8px!important}\n .xposim-explorer .xposim-theme-selector {justify-self:end}\n .xposim-explorer .xposim-runtime {width:100%!important}\n}\n@container xp-explorer (max-width:540px) {\n .xposim-explorer .xposim-run-metrics {grid-template-columns:1fr}\n .xposim-explorer .xposim-run-metrics>div:nth-child(odd) {border-right:0}\n .xposim-explorer .xposim-run-metrics>div:last-child {border-bottom:0}\n .xposim-explorer .xposim-selection-row {display:grid!important;grid-template-columns:34px minmax(0,1fr) minmax(0,1fr) 34px;gap:8px 4px}\n .xposim-explorer .xposim-previous-plane {grid-column:1;grid-row:1}\n .xposim-explorer .xposim-plane-slider {grid-column:2/4;grid-row:1;width:100%!important;min-width:0!important}\n .xposim-explorer .xposim-next-plane {grid-column:4;grid-row:1}\n .xposim-explorer .xposim-peak-plane {grid-column:1/3;grid-row:2}\n .xposim-explorer .xposim-z-input {grid-column:3/5;grid-row:2;width:100%!important}\n .xposim-explorer .xposim-plane-label {grid-column:1/-1;grid-row:3;width:auto!important;justify-self:end}\n}\n\n.xposim-explorer .xposim-plot-compact {display:none!important}\n@container xp-explorer (max-width:540px) {\n .xposim-explorer .xposim-plot-wide {display:none!important}\n .xposim-explorer .xposim-plot-compact {display:block!important}\n}\n</style>\n'
UI_CSS=UI_CSS.replace('</style>', '''
.xp-parameter-overview,.xp-geometry-fields,.xp-material-fields {
 grid-template-columns:minmax(0,1.55fr) minmax(0,1fr);gap:9px;align-items:start;
}
.xp-geometry-fields {grid-template-columns:minmax(0,1.22fr) minmax(0,1fr)}
.xposim-parameters .xp-parameter-overview>div {height:100%}
.xposim-parameters>.xposim-section + .xposim-section {margin-top:7px}
.xposim-parameters .xp-detector-toggle {min-height:30px;margin:4px 0}
.xposim-parameters .xp-detector-toggle label {white-space:normal}
.xposim-parameters .xposim-field-caption {min-height:22px!important}
.xposim-parameters .xp-source-grid {grid-template-columns:repeat(4,minmax(0,1fr))}
.xposim-parameters .xp-source-grid .xposim-field-source {grid-column:span 2}
.xposim-parameters .xp-geometry-fields .xp-control-grid {grid-template-columns:repeat(2,minmax(0,1fr))}
.xposim-parameters .xp-parameter-overview .xposim-analytical-list,
.xposim-parameters .xp-material-fields .xposim-analytical-list {margin:0}
.xposim-parameters .xp-parameter-overview .xposim-analytical-list dd,
.xposim-parameters .xp-material-fields .xposim-analytical-list dd {font-size:15px}
.xposim-parameters .xp-parameter-overview .xposim-analytical-list .xposim-magnification {grid-template-columns:minmax(0,1fr) auto}
.xposim-parameters .xp-parameter-overview .xposim-analytic-status {min-height:23px!important}
.xp-optical-layout,.xp-element-view {min-width:0;background:var(--xp-bg);border:1px solid var(--xp-grid);border-radius:6px}
.xp-optical-layout svg,.xp-element-view svg {display:block;width:100%;height:auto}
.xp-element-view svg {max-height:224px}
.xp-optical-layout text,.xp-element-view text {fill:var(--xp-fg);font-family:inherit;dominant-baseline:alphabetic}
.xp-layout-label {font-size:17px;font-weight:650}
.xp-optical-layout .xp-layout-small,.xp-element-view .xp-layout-small {font-size:14px;fill:var(--xp-secondary)}
.xp-layout-axis,.xp-layout-dimension,.xp-layout-radius {fill:none;stroke:var(--xp-secondary);stroke-width:1.15}
.xp-optical-layout .xp-stack-count {font-weight:650;fill:var(--xp-fg)}
.xp-crossed-layout .xp-crossed-source {font-size:13px}
.xposim-parameters .xp-crossed-mobile {display:none}
@container xp-section (max-width:430px) {
 .xp-crossed-layout .xp-crossed-secondary {display:none}
 .xp-crossed-layout .xp-layout-small.xp-stack-count {font-size:18px}
 .xposim-parameters .xp-crossed-mobile {display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:4px 12px;margin:5px 0}
 .xp-crossed-mobile>div {display:flex;flex-wrap:wrap;gap:2px 8px;border-bottom:1px solid var(--xp-grid);padding:4px 0;font-size:12px}
 .xp-crossed-mobile dt,.xp-crossed-mobile dd {width:auto;float:none;margin:0}
 .xp-crossed-mobile dd {font-weight:650;white-space:nowrap}
}
.xp-layout-axis {stroke-dasharray:4 3;opacity:.65}
.xp-layout-radius {stroke-dasharray:3 2}
.xp-layout-source-shape {fill:#e5a344;fill-opacity:.13;stroke:#b47527;stroke-width:1.35}
.xp-layout-ray {fill:none;stroke:#d19843;stroke-width:1.65;stroke-linejoin:round}
.xp-layout-source {fill:#d19843;stroke:var(--xp-surface);stroke-width:1.4}
.xp-layout-lens {fill:#647da4;fill-opacity:.26;stroke:#6581ad;stroke-width:1.35}
.xp-radius-marker {fill:none;stroke:#6581a4;stroke-width:1.35}
.xp-dimension-arrow {fill:var(--xp-secondary);stroke:var(--xp-secondary);stroke-width:.65}
.xp-round-back {fill:light-dark(#8ea5b9,#546d86)}
.xp-round-cap {fill:light-dark(#8399ad,#55728a);stroke:#566d81;stroke-width:1.2}
.xp-round-aperture {fill:light-dark(#c5d4e0,#829bb2);stroke:#566d81;stroke-width:1}
.xp-round-case-top {fill:light-dark(#b7c8d5,#7791a7)}
.xp-round-case-light {fill:light-dark(#a9bccb,#69839b)}
.xp-round-case-mid {fill:light-dark(#9aafc1,#5e7890)}
.xp-round-case-bottom {fill:light-dark(#8199ae,#526b84)}
.xp-round-cut {fill:light-dark(#d4e0ea,#7c92ae);stroke:#829ab0;stroke-width:.95}
.xp-round-window-edge {fill:none;stroke:light-dark(#5e768b,#b6c9dc);stroke-width:1.1}
.xp-round-internal-ray {fill:none;stroke:#d19843;stroke-width:1.1;stroke-dasharray:4 3;opacity:.70}
.xp-round-cavity-near {fill:light-dark(#b5c5d8,#728cad);stroke:#667d96;stroke-width:.9}
.xp-round-cavity-far {fill:light-dark(#8fa6bf,#536c8a);stroke:#667d96;stroke-width:.9}
.xp-round-mesh {fill:none;stroke:light-dark(#526e8b,#b0c5dd);stroke-width:.85;opacity:.78}
.xp-layout-focus,.xp-layout-focus-line {stroke:light-dark(#167c74,#4ec9b8);fill:light-dark(#167c74,#4ec9b8);stroke-width:2.5}
.xp-layout-caption {font-size:12px;color:var(--xp-secondary);padding:5px 0 0}
@container xp-parameters (max-width:780px) {
 .xp-parameter-overview,.xp-geometry-fields,.xp-material-fields {grid-template-columns:1fr}
 .xp-element-view {max-width:520px}
 .xp-optical-layout .xp-layout-label {font-size:20px}
 .xp-optical-layout .xp-layout-small {font-size:15px}
}
@container xp-section (max-width:360px) {
 .xposim-parameters .xp-parameter-overview .xposim-analytical-list .xposim-magnification {grid-template-columns:1fr}
}
</style>''')
UI_CSS=UI_CSS.replace('</style>', '''.xp-layout-slab {fill:var(--xp-secondary);fill-opacity:.15;stroke:var(--xp-secondary);stroke-width:1.35}
.xp-layout-cavity,.xp-layout-hidden {fill:none;stroke:var(--xp-secondary);stroke-width:1.25}
.xp-layout-hidden {stroke-dasharray:3 2;opacity:.43}
.xp-crl-hatch {fill:url(#xp-crl-hatch);stroke:none}
.xp-dimension-symbol {font-weight:650}
.xp-layout-hatch {fill:none;stroke:var(--xp-secondary);stroke-width:.6;opacity:.34}
.xp-layout-source-shape {fill:url(#xp-gaussian);fill-opacity:1;stroke:#b47caa;stroke-width:1}
</style>''')
UI_CSS=UI_CSS.replace('</style>', '''
.xposim-parameters .xp-parameter-overview .xposim-analytical-list>div {
 grid-template-columns:minmax(0,1fr) minmax(116px,auto);gap:3px 8px;
}
.xposim-parameters .xp-parameter-overview .xposim-analytical-list .xposim-magnification {grid-template-columns:1fr}
.xposim-parameters .xposim-magnification-axis {
 display:grid;grid-template-columns:minmax(0,1fr) auto;align-items:baseline;gap:4px 10px;text-align:right
}
.xposim-parameters .xposim-magnification-axis .xposim-axis-name {white-space:normal}
.xposim-parameters .xposim-magnification-axis .xposim-number {grid-column:2}
.xposim-parameters .xposim-magnification dd {padding:3px 0 2px 9px;border-left:2px solid var(--xp-grid)}
.xp-crossed-x {fill:none;stroke:light-dark(#217e94,#6bd4e2);stroke-width:1.4}
.xp-crossed-y {fill:none;stroke:light-dark(#a86440,#efb481);stroke-width:1.4}
.xp-crossed-plane {fill:none;stroke-dasharray:4 3;stroke-width:1.4}
.xp-crossed-status {margin-top:5px;padding:5px 8px;border-left:3px solid light-dark(#926131,#d5ab74);font-size:13px}
.xp-crossed-status[data-aligned=true] {border-color:light-dark(#176d60,#75d6c0)}
.xposim-parameters .xp-optical-layout svg {max-height:234px}
@container xp-parameters (max-width:620px) {
 .xposim-parameters .xp-control-grid {grid-template-columns:repeat(2,minmax(0,1fr))!important}
 .xp-control-grid .xposim-field-source,.xp-control-grid .xposim-field-source_method,
 .xp-control-grid .xposim-field-focusing,.xp-control-grid .xposim-field-material_mode,
 .xp-control-grid .xposim-field-material,.xp-control-grid .xposim-field-roi_mode,
 .xp-control-grid .xposim-field-y_material_mode,.xp-control-grid .xposim-field-y_material {grid-column:1/-1}
}
</style>''')
ui_style=widgets.HTML(UI_CSS,layout=widgets.Layout(height='0px',margin='0',overflow='visible'))
parameter_editor=widgets.VBox([ui_style,source_section,geometry_section,material_section,second_stack_section,spacing_section,detector_toggle,roi_section,overview],
    layout=widgets.Layout(width='100%',max_width='1280px',margin='0 auto',gap='8px',overflow='visible'))
if 'current_theme' in globals():parameter_editor.add_class('xposim-'+current_theme)
parameter_editor.add_class('xposim-parameters')
display(parameter_editor)


In [ ]:
#@title 3 · Results — press Compute after entering parameters above
import base64
import gc
import io
import time
import tempfile
import textwrap
from datetime import datetime, timezone
from pathlib import Path
from contextlib import ExitStack
from matplotlib.ticker import FixedLocator, FuncFormatter, MaxNLocator
from matplotlib.colors import Normalize
from matplotlib.colorbar import ColorbarBase
from matplotlib import patheffects, font_manager

if 'controls' not in globals() or 'parameter_editor' not in globals():
    raise RuntimeError('Run the CRL and source parameters cell above before Results.')

# Reinitialization preserves inputs and waits for an explicit calculation.
_previous_theme=globals().get('theme_selector')
_theme_choice=_previous_theme.value if _previous_theme is not None else 'Auto'
_previous_state=globals().get('state',{})
if _previous_state.get('plane_update_handle') is not None:
    _previous_state['plane_update_handle'].cancel()
for name in ('slider','z_input','previous_button','next_button','peak_button','plane_label'):
    widget=_previous_state.get(name)
    if widget is not None:
        widget.unobserve_all()
        widget.close()
for name in ('explorer_root','compute_button','plane_image','scan_image','plane_compact_image','scan_compact_image','plane_view','progress_view','shared_scale','color_scale','results_notice','result_style','theme_selector','plane_heading','plane_metrics','scan_heading','source_comparison','report_button','report_message','report_panel'):
    old=globals().get(name)
    if old is not None:
        old.close()
state={'result':None,'slider':None,'handler':None,'plane_updates':0,'error':None,
       'z_unit_scale':1e-3,'scan_axes':None}
CMAP=plt.get_cmap('CMRmap')
_available_fonts={font.name for font in font_manager.fontManager.ttflist}
PLOT_FONT=next(name for name in ('Consolas','DejaVu Sans Mono','Liberation Mono') if name in _available_fonts)
INTENSITY_COLOR='#355f85'
PROFILE_COLOR='#355f85'
PROFILE_COLORS=(PROFILE_COLOR,'#788494')
WIDTH_COLORS=('#3d7665','#7aa494')
CURSOR_COLOR='#1ad1c1'
PANEL_SIZE=(6.4,5.45)
PLANE_SIZE=PANEL_SIZE
THEMES={
    'light':dict(surface='#ffffff',background='#f3f6fa',foreground='#172235',secondary='#465b74',border='#9aaabe',grid='#d6dee9',blue='#355f85',green='#3d7665',other='#788494',cursor='#119f93'),
    'dark':dict(surface='#222a35',background='#181d25',foreground='#f1f5fa',secondary='#c0ccdd',border='#64768f',grid='#3c495c',blue='#93c4ff',green='#91ddbb',other='#b5becd',cursor='#58e5d6')}
palette=THEMES[current_theme]
theme_selector=widgets.Dropdown(options=['Auto','Dark','Light'],value=_theme_choice,description='Theme',
    style={'description_width':'initial'},layout=widgets.Layout(width='160px',min_width='0'))
theme_selector.add_class('xposim-theme-selector')
compute_button=widgets.Button(description='Compute',button_style='primary',icon='play',
                              layout=widgets.Layout(width='125px'))
compute_button.style.button_color=INTENSITY_COLOR
report_button=widgets.Button(description='Download PDF report',icon='download',disabled=True,
    layout=widgets.Layout(width='auto',min_width='185px'))
report_button.tooltip='Complete a calculation to export a report.'
report_message=widgets.HTML(layout=widgets.Layout(min_width='0'))
report_panel=widgets.HBox([report_button,report_message],layout=widgets.Layout(
    width='100%',flex_flow='row wrap',align_items='center',gap='8px',margin='6px 0 0'))
report_panel.add_class('xposim-report-panel')
status=widgets.HTML('<span style="font-size:13px">Set parameters, then press Compute.</span>',layout=widgets.Layout(flex='1 1 0%',min_width='0'))
status.add_class('xposim-runtime')
summary=widgets.HTML(layout=widgets.Layout(width='100%',overflow='visible',margin='0'))
source_comparison=widgets.HTML(layout=widgets.Layout(width='100%',display='none'))
source_comparison.add_class('xposim-source-comparison')
results_notice=widgets.HTML(layout=widgets.Layout(width='100%',display='none',overflow='visible',margin='0'))
progress_view=widgets.HTML(layout=widgets.Layout(width='100%',display='none'))
progress_view.add_class('xposim-progress')
scan_image=widgets.Image(format='png',width='896',height='763',layout=widgets.Layout(width='100%',max_width='100%',min_width='0',height='auto',margin='0'))
plane_image=widgets.Image(format='png',width='896',height='763',layout=widgets.Layout(width='100%',max_width='100%',min_width='0',height='auto',margin='0'))
scan_compact_image=widgets.Image(format='png',width='504',height='1176',layout=widgets.Layout(width='100%',max_width='100%',height='auto',margin='0'))
plane_compact_image=widgets.Image(format='png',width='504',height='700',layout=widgets.Layout(width='100%',max_width='100%',height='auto',margin='0'))
for widget in (scan_image,plane_image):widget.add_class('xposim-plot-wide')
for widget in (scan_compact_image,plane_compact_image):widget.add_class('xposim-plot-compact')
plane_controls=widgets.VBox(layout=widgets.Layout(width='100%',display='none'))
details_text=widgets.HTML(layout=widgets.Layout(width='100%',overflow='visible',margin='0'))
plane_heading=widgets.HTML(layout=widgets.Layout(width='100%',overflow='visible',margin='0'))
scan_heading=widgets.HTML('<div class="xposim-panel-heading">Propagation along z</div>',layout=widgets.Layout(width='100%',overflow='visible',margin='0'))
plane_metrics=widgets.HTML(layout=widgets.Layout(display='none'))
plane_metrics.add_class('xposim-plane-readouts')
details=widgets.Accordion(children=[details_text],selected_index=None)
details.set_title(0,'Model, mesh and diagnostics')


def compact_number(value):
    return decimal_number(float(value))


TICK_FORMATTER=FuncFormatter(lambda value,position:compact_number(value))


def engineering(values, unit=''):
    """Keep ticks compact and move a common power of ten to the axis label."""
    finite=np.asarray(values,dtype=float)
    finite=finite[np.isfinite(finite)]
    maximum=float(np.max(abs(finite))) if finite.size else 0.0
    exponent=0
    if maximum>=80:
        exponent=int(np.ceil(np.log10(maximum/80)))
    elif 0<maximum<0.01:
        exponent=int(np.floor(np.log10(maximum)))
    multiplier=10.0**exponent
    suffix=(' '+r'$\times\,10^{'+str(exponent)+'}$') if exponent else ''
    return multiplier,unit+suffix


def length_units(values):
    finite=np.asarray(values,dtype=float)
    finite=finite[np.isfinite(finite)]
    maximum=float(np.max(abs(finite))) if finite.size else 0
    scale,label=(1.0,'m') if maximum>=0.1 else ((1e-3,'mm') if maximum>=1e-4 else ((1e-6,'µm') if maximum>=1e-7 else (1e-9,'nm')))
    power,suffix=engineering(finite/scale,label)
    return scale*power,suffix


def axial_units(result):
    coordinates=result.z_m
    step=result.scan_step_m
    # Use a smaller unit when two decimal places would hide distinct planes.
    if coordinates[-1]<1e-7 or step<=1e-8*(1+1e-9):
        return 1e-9,'nm'
    if coordinates[-1]<1e-4 or step<=1e-5*(1+1e-9):
        return 1e-6,'µm'
    if coordinates[-1]>=.1 and step>.01*(1+1e-9):
        return 1.,'m'
    return 1e-3,'mm'


def interval_units(value):
    return (1e-3,'mm') if abs(value)>=1e-5*(1-1e-9) else (
        (1e-6,'µm') if abs(value)>=1e-8 else (1e-9,'nm'))


def axial_interval(value):
    if value is None or not np.isfinite(value):
        return 'unresolved'
    scale,unit=interval_units(value)
    return compact_number(value/scale)+' '+unit


def fixed_axes(ax, x=True, y=True):
    for axis,enabled in ((ax.xaxis,x),(ax.yaxis,y)):
        if enabled:
            axis.set_major_locator(MaxNLocator(nbins=4,min_n_ticks=3))
            axis.set_major_formatter(TICK_FORMATTER)
    ax.tick_params(labelsize=10.5)
    for spine in ax.spines.values():
        spine.set_color(palette['border'])
    ax.set_facecolor(palette['surface'])


def label_narrow_map_axis(ax):
    """Keep coordinate marks distinct on thin or tightly cropped maps."""
    width,height=np.ptp(ax.get_xlim()),np.ptp(ax.get_ylim())
    for axis,(low,high),thin in ((ax.xaxis,ax.get_xlim(),width<.25*height),
                                  (ax.yaxis,ax.get_ylim(),height<.25*width)):
        span=high-low
        ticks=np.asarray(axis.get_majorticklocs())
        ticks=ticks[(ticks>=low)&(ticks<=high)]
        distinct=len({compact_number(value) for value in ticks})==len(ticks)
        if ticks.size>=2 and distinct and not thin:
            continue
        if ticks.size>=2 and ticks[-1]-ticks[0]>=.56*span:
            ticks=ticks[[0,-1]]
        else:
            ticks=[low+.12*span,high-.12*span]
        precision=max(2,1-math.floor(math.log10(ticks[1]-ticks[0])))

        def format_tick(value,_,digits=precision):
            if digits>12:
                return np.format_float_scientific(value,unique=True,trim='-')
            label=f'{value:.{digits}f}'.rstrip('0').rstrip('.')
            return '0' if label=='-0' else label

        axis.set_major_locator(FixedLocator(ticks))
        axis.set_major_formatter(FuncFormatter(format_tick))
        if thin:
            # Let text extend outwards from the two marks on a thin strip.
            for index,tick in enumerate(axis.get_major_ticks(2)):
                for label in (tick.label1,tick.label2):
                    if axis is ax.yaxis:label.set_verticalalignment('top' if index==0 else 'bottom')
                    else:label.set_horizontalalignment('right' if index==0 else 'left')


def set_render_value(widget,value):
    """Stage one complete frame before publishing any of its widgets."""
    frame=state.get('render_values')
    if frame is None:widget.value=value
    else:frame[widget]=value


def plot_png(figure,compact_widget,compact_size):
    stream=io.BytesIO()
    figure.savefig(stream,format='png',dpi=140,facecolor=palette['surface'])
    wide_size=figure.get_size_inches().copy()
    figure.set_size_inches(*compact_size)
    compact=io.BytesIO()
    compact_titles=[]
    for ax in figure.axes:
        title=ax.get_title()
        if title in ('Intensity distribution near the focus','Transverse intensity · shared ROI'):
            compact_titles.append((ax,title))
            ax.set_title(title.replace(' near','\nnear').replace(' · shared','\nShared'))
    # Keep numeric extrema inside narrow axes without rounding their values.
    figure.canvas.draw()
    renderer=figure.canvas.get_renderer()
    label_positions=[]
    for ax in figure.axes:
        for artist in ax.texts:
            if artist.get_gid()!='xposim-extremum':continue
            bounds=artist.get_bbox_patch().get_window_extent(renderer)
            shift=min(0,ax.bbox.x1-4-bounds.x1)+max(0,ax.bbox.x0+4-bounds.x0)
            if shift:
                x,y=artist.get_position()
                label_positions.append((artist,(x,y)))
                artist.set_position((x+shift*72/figure.dpi,y))
    figure.savefig(compact,format='png',dpi=140,facecolor=palette['surface'])
    set_render_value(compact_widget,compact.getvalue())
    for ax,title in compact_titles:ax.set_title(title)
    for artist,position in label_positions:artist.set_position(position)
    figure.set_size_inches(*wide_size)
    figure.canvas.draw()
    plt.close(figure)
    return stream.getvalue()


def metric_value(value, unit=''):
    if value is None or not np.isfinite(value):
        return 'unresolved'
    exponent=0
    if value and (abs(value)<0.1 or abs(value)>=1e4):
        exponent=int(np.floor(np.log10(abs(value))))
    text=compact_number(value/10.0**exponent)
    if exponent:
        text+=f' × 10<sup>{exponent}</sup>'
    return text+((' '+html.escape(unit)) if unit else '')


def distance_value(value):
    # Cards use physical units instead of embedding Matplotlib math markup.
    scale,unit=(1.,'m') if abs(value)>=.1 else ((1e-3,'mm') if abs(value)>=1e-4 else ((1e-6,'µm') if abs(value)>=1e-7 else (1e-9,'nm')))
    return metric_value(value/scale,unit)


def width_value(value):
    if value is None or not np.isfinite(value):
        return 'unresolved'
    scale,unit=(1e-9,'nm') if abs(value)<1e-6 else (1e-6,'µm')
    return metric_value(value/scale,unit)


def axis_values(values, formatter):
    values=tuple(values)
    if len(values)==1:
        return formatter(values[0])
    return ' · '.join('<span style="white-space:nowrap">'+axis+': '+formatter(value)+'</span>' for axis,value in zip(('x','y'),values))


def plot_style():
    return {'savefig.bbox':None,'font.family':'monospace','font.monospace':[PLOT_FONT],
        'mathtext.fontset':'custom','mathtext.rm':PLOT_FONT,
        'mathtext.it':PLOT_FONT+':italic','mathtext.bf':PLOT_FONT+':bold',
        'mathtext.cal':PLOT_FONT,'mathtext.sf':PLOT_FONT,'mathtext.tt':PLOT_FONT,
        'font.size':11,'axes.titlesize':12,'axes.labelsize':11,
        'figure.facecolor':palette['surface'],'axes.facecolor':palette['surface'],
        'text.color':palette['foreground'],'axes.labelcolor':palette['foreground'],
        'xtick.color':palette['secondary'],'ytick.color':palette['secondary'],
        'grid.color':palette['grid'],'axes.edgecolor':palette['border']}


def set_plot_palette(theme):
    global palette,PROFILE_COLOR,PROFILE_COLORS,WIDTH_COLORS,INTENSITY_COLOR,CURSOR_COLOR
    palette=THEMES[theme]
    PROFILE_COLOR=INTENSITY_COLOR=palette['blue']
    PROFILE_COLORS=(palette['blue'],palette['other'])
    WIDTH_COLORS=(palette['green'],palette['other']);CURSOR_COLOR=palette['cursor']


def apply_theme(refresh=True,*,detect=False):
    global current_theme
    chosen=(_detect_theme() if detect else current_theme) if theme_selector.value=='Auto' else theme_selector.value.lower()
    changed=chosen!=current_theme
    current_theme=chosen
    set_plot_palette(chosen)
    for widget in (parameter_editor,globals().get('explorer_root')):
        if widget is not None:
            widget.remove_class('xposim-light');widget.remove_class('xposim-dark')
            widget.add_class('xposim-'+current_theme)
    if state.get('slider') is not None:state['slider'].style.handle_color=CURSOR_COLOR
    if changed:state.pop('scan_cursor_m',None)
    if refresh and state.get('result') is not None:
        pending=state.get('pending_selection')
        target=pending[1] if pending is not None and pending[0] is state['result'] else state['slider'].value
        select_plane(target)


def profile_is_dashed(axis,result=None):
    if result is None:result=state.get('result')
    one_axis_2d=(result is not None and result.configuration.grid_dimension==2
        and result.configuration.geometry=='one-dimensional')
    return False if one_axis_2d else axis=='y'


def axis_badge(axis):
    color=PROFILE_COLORS[0 if axis=='x' else 1]
    line='dashed' if profile_is_dashed(axis) else 'solid'
    return '<span class="xposim-axis-badge"><span aria-hidden="true" style="border-top:2px '+line+' '+color+'"></span>'+axis+'</span>'


def axis_table(headers,rows):
    head='<thead><tr>'+''.join('<th scope="col">'+html.escape(label)+'</th>' for label in headers)+'</tr></thead>' if headers else ''
    body='<tbody>'+''.join('<tr>'+''.join('<td>'+value+'</td>' for value in row)+'</tr>' for row in rows)+'</tbody>'
    return '<table class="xposim-axis-table">'+head+body+'</table>'


def metric_box(label,unit,value):
    return '<section class="xposim-metric-box"><div class="xposim-metric-title">'+html.escape(label)+'<span class="xposim-unit">'+html.escape(unit)+'</span></div><div class="xposim-metric-value">'+value+'</div></section>'


def z_text(value):
    return compact_number(value/state['z_unit_scale'])


def depth_label(result):
    widths=result.depth_of_focus_m
    labels=[axial_interval(value) for value in widths]
    if len(labels)==1 or labels[0]==labels[-1]:
        return labels[0]
    return ' · '.join(axis+' '+label for axis,label in zip(result.fwhm_axes,labels))


def source_width_comparison(result):
    point=result.point_source_fwhm_at_peak_m
    if point is None:
        return ''
    rows=[]
    for axis,initial,final,factor in zip(result.fwhm_axes,point,np.atleast_1d(result.fwhm_at_peak_m),result.source_broadening_factor):
        detail='Unresolved' if factor is None else width_value(initial)+' → '+width_value(final)+' · × '+compact_number(factor)
        rows.append('<span><b>'+axis+'</b> '+detail+'</span>')
    sparse=[axis for axis in ('x','y') if any(message.startswith('source_quadrature_'+axis+':')
                                             for message in result.diagnostics)]
    if sparse:
        rows.append('<span><b>Check source sampling along '+', '.join(sparse)+'</b></span>')
    return ('<div class="xp-source-widths"><b>Finite-source broadening at the reference plane</b>'
            '<span>Point-source FWHM → Gaussian-source FWHM</span>'+''.join(rows)+'</div>')


def update_summary(result,plane):
    alignment=result.crossed_alignment
    scale,unit=axial_units(result)
    peak_kind='sampled' if result.numerical_peak.sampled else 'numerical'
    headers=[('Peak-intensity position ('+peak_kind+')',unit),('Transmission','%')]
    values=[compact_number(result.peak_z_m/scale),compact_number(result.transmission*100)]
    if alignment is None:
        aperture_size='diameter' if result.configuration.geometry=='two-dimensional' else 'width'
        aperture=state['analytical_result'].kohn_effective_aperture_m
        headers.append(('Effective aperture ('+aperture_size+')','µm'))
        values.append('∞' if math.isinf(aperture) else compact_number(aperture*1e6))
    else:
        headers.append(('Equivalent aperture area','µm²'))
        values.append(compact_number(result.equivalent_aperture_area_m2*1e12))
        match=result.equal_fwhm_plane
        headers.append(('Equal-FWHM position · from y exit',unit if match is not None else ''))
        values.append(compact_number(match.z_m/scale) if match is not None else 'Unresolved')
    note=' · central point-source reference' if result.configuration.source_type=='gaussian' and alignment is None else ''
    if alignment is not None:
        note+=' · coincident paraxial images' if alignment.aligned else ' · no common paraxial image plane'
    if result.configuration.detector_roi_m is not None:note+=' · manual calculation region'
    set_render_value(summary,'<section class="xposim-run"><div class="xposim-group-title">Simulation summary'+note+'</div><dl class="xposim-run-metrics" style="--xp-metric-count:'+str(len(headers))+'">'+''.join('<div><dt>'+label+'</dt><dd><span class="xposim-metric-number">'+value+'</span><span class="xposim-unit">'+unit+'</span></dd></div>' for (label,unit),value in zip(headers,values))+'</dl></section>')
    band='<span class="xp-panel-note">Shading marks where the beam FWHM is within 41% of its minimum.</span>' if any(result.depth_of_focus_bounds_m) else ''
    if alignment is not None:
        minima=result.sampled_fwhm_minimum_z_m
        minima_text=' · '.join(axis+' '+(compact_number(z/scale)+' '+unit if z is not None else 'unresolved')
                                for axis,z in zip(result.fwhm_axes,minima))
        band+='<span class="xp-panel-note">Sampled FWHM minima z · '+minima_text+'</span>'
    set_render_value(scan_heading,'<div class="xposim-panel-heading">Propagation along z'
        '<span class="xp-depth-readout">Depth of focus · '+depth_label(result)+'</span>'+band+'</div>')
    analysis=state.get('analytical_result')
    lengths=analysis.transverse_coherence_length_m(plane.z_m) if analysis is not None and alignment is None else None
    coherence=''
    if lengths is not None:
        coherence=('<span class="xp-panel-note" title="Slit separation at half fringe visibility for a centered pair of slits; Gaussian-pupil estimate">'
                   'Transverse coherence length · '
                   +coherence_values(lengths,lambda v:'∞' if math.isinf(v) else width_value(v))+'</span>')
    set_render_value(plane_heading,'<div class="xposim-panel-heading">Selected transverse plane<span class="xposim-selected-z">z = '+z_text(plane.z_m)+' <span style="font-size:13px;font-weight:400">'+html.escape(state['z_unit'])+'</span></span>'+coherence+'</div>')
    set_render_value(plane_metrics,'')


def outside_legend(ax):
    # Constrained layout reserves this space, keeping keys clear of every curve.
    return ax.legend(frameon=False,fontsize=10,loc='lower left',
        bbox_to_anchor=(0,1.01),borderaxespad=0,ncol=2,handlelength=2,columnspacing=1.4)


def finite_extreme(values,mode,reference_index):
    values=np.asarray(values,dtype=float)
    indices=np.flatnonzero(np.isfinite(values))
    if not indices.size:return None
    best=np.max(values[indices]) if mode=='max' else np.min(values[indices])
    ties=indices[values[indices]==best]
    return int(ties[np.argmin(abs(ties-reference_index))])


def mark_scan_extrema(result,peak_ax,width_ax,width_arrays,zscale,peakscale,widthscale,*,printed=False):
    face,ink=('white','#172b42') if printed else (palette['surface'],palette['foreground'])
    fontsize=7 if printed else 9
    extrema=[]
    index=finite_extreme(result.peak_intensity,'max',result.peak_index)
    if index is not None:
        x,y=result.z_m[index]/zscale,result.peak_intensity[index]/peakscale
        offset=-7 if x>np.mean(result.z_m)/zscale else 7
        if printed:peak_ax.plot([x],[y],'o',color='#355f85',ms=2.9,zorder=5)
        label_artist=peak_ax.annotate('max: '+plot_metric(result.peak_intensity[index],'a.u.'),xy=(x,y),xytext=(offset,-4 if printed else -8),textcoords='offset points',
            ha='right' if offset<0 else 'left',va='top',fontsize=fontsize,color=ink,
            bbox=dict(facecolor=face,alpha=.9,edgecolor='none',pad=1))
        label_artist.set_gid('xposim-extremum');label_artist.set_in_layout(False)
        extrema.append(dict(kind='max',axis='intensity',index=index,z_m=float(result.z_m[index]),value=float(result.peak_intensity[index])))
    widths=[]
    for axis,values in zip(('x','y'),width_arrays):
        index=finite_extreme(values,'min',result.peak_index)
        if index is not None:widths.append((axis,index,float(values[index])))
    coincident=(len(widths)==2 and widths[0][1]==widths[1][1]
        and np.isclose(widths[0][2],widths[1][2],rtol=1e-6,atol=0))
    separate_minima=len(widths)==2 and not coincident
    grouped_labels=[]
    for n,(axis,index,value) in enumerate(widths):
        extrema.append(dict(kind='min',axis=axis,index=index,z_m=float(result.z_m[index]),value=value))
        if coincident and n==1:continue
        x,y=result.z_m[index]/zscale,value/widthscale
        label='min x,y' if coincident else ('min '+axis if len(width_arrays)>1 else 'min')
        unit_scale,unit=(1e-9,'nm') if value<1e-6 else (1e-6,'µm')
        label+=': '+plot_metric(value/unit_scale,unit)
        if separate_minima:
            grouped_labels.append(label)
            color=width_ax.lines[n].get_color()
            width_ax.plot([x],[y],'o',color=color,ms=2.8 if printed else 3.4,zorder=5)
            continue
        offset=-7 if x>np.mean(result.z_m)/zscale else 7
        if printed:width_ax.plot([x],[y],'o',color='#356a57',ms=2.8,zorder=5)
        label_artist=width_ax.annotate(label,xy=(x,y),xytext=(offset,4 if printed else 6),textcoords='offset points',
            ha='right' if offset<0 else 'left',va='bottom',fontsize=fontsize,color=ink,
            bbox=dict(facecolor=face,alpha=.9,edgecolor='none',pad=1))
        label_artist.set_gid('xposim-extremum');label_artist.set_in_layout(False)
    if grouped_labels:
        label_artist=width_ax.text(.98,.94,'\n'.join(grouped_labels),transform=width_ax.transAxes,
            ha='right',va='top',multialignment='left',fontsize=fontsize,color=ink,
            bbox=dict(facecolor=face,alpha=.9,edgecolor='none',pad=1),zorder=6)
        label_artist.set_gid('xposim-extremum');label_artist.set_in_layout(False)
    return extrema


def shade_depth_of_focus(ax,result,zscale,colors):
    previous=None
    for index,bounds in enumerate(result.depth_of_focus_bounds_m):
        if bounds is None or (previous is not None and np.allclose(bounds,previous,rtol=1e-8,atol=0)):
            continue
        ax.axvspan(bounds[0]/zscale,bounds[1]/zscale,color=colors[index],
                   alpha=.18,linewidth=0,zorder=0)
        previous=bounds


def axial_label(result,unit):
    exit_plane='y CRL exit' if result.crossed_alignment is not None else 'CRL exit'
    return f'z from {exit_plane} ({unit})'


def longitudinal_sections(result):
    sections=[('x',result.x_m,result.longitudinal_intensity)]
    if result.longitudinal_intensity_y is not None:
        sections.append(('y',result.y_m,result.longitudinal_intensity_y))
    return sections


def draw_longitudinal_section(ax,section,z,reference,bounds=None):
    axis,coordinate,data=section
    scale,unit=length_units(coordinate)
    mesh=ax.pcolormesh(z,coordinate/scale,data.T/reference,shading='auto',cmap=CMAP,
                       norm=Normalize(0,1),rasterized=True)
    ax.set_ylabel(f'{axis} ({unit})')
    if bounds is not None:
        ax.set_ylim(*(value/scale for value in bounds))
    return mesh


def render_scan(result, selected_plane):
    zscale,zunit=axial_units(result)
    z=result.z_m/zscale
    sections=longitudinal_sections(result)
    peakscale,peakunit=engineering(result.peak_intensity,'a.u.')
    xwidth=result.fwhm_m if result.fwhm_m.ndim==1 else result.fwhm_m[:,0]
    width_arrays=[xwidth]
    if len(result.fwhm_axes)==2:
        width_arrays.append(result.fwhm_m[:,1])
    widthscale,widthunit=length_units(np.concatenate(width_arrays))
    with plt.rc_context(plot_style()):
        figure=plt.figure(figsize=PANEL_SIZE,layout='constrained')
        count=len(sections)
        ratios=[2.,1,1] if count==1 else [1.35,1.35,1,1]
        spec=figure.add_gridspec(count+2,1,height_ratios=ratios,hspace=.04)
        map_axes=[figure.add_subplot(spec[0,0])]
        if count==2:
            map_axes.append(figure.add_subplot(spec[1,0],sharex=map_axes[0]))
        peak_ax=figure.add_subplot(spec[count,0],sharex=map_axes[0])
        width_ax=figure.add_subplot(spec[count+1,0],sharex=map_axes[0])
        top=max(result.intensity_reference,np.finfo(float).tiny)
        manual_roi=result.configuration.detector_roi_m
        limits=(manual_roi,) if result.configuration.grid_dimension==1 else manual_roi
        for index,(ax,section) in enumerate(zip(map_axes,sections)):
            mesh=draw_longitudinal_section(ax,section,z,top,None if limits is None else limits[index])
            if count==2:
                ax.set_title(f'{section[0]}–z intensity',loc='left',fontsize=11)
            else:
                title='Intensity in sampled region' if manual_roi is not None else 'Intensity distribution near the focus'
                ax.set_title(title)
        if manual_roi is not None and result.configuration.scan_range_m is not None:
            map_axes[0].set_xlim(*(value/zscale for value in result.configuration.scan_range_m))
        if result.configuration.grid_dimension==1:
            cax=map_axes[0].inset_axes([1.025,0,.025,1])
            cb=figure.colorbar(mesh,cax=cax);cb.set_ticks([0,.5,1])
            cb.ax.yaxis.set_major_formatter(TICK_FORMATTER);cb.ax.tick_params(labelsize=9,colors=palette['secondary'])
            cb.outline.set_visible(False);cb.ax.set_title('I / Iref',fontsize=9,color=palette['foreground'])
        peak_ax.plot(result.z_m/zscale,result.peak_intensity/peakscale,color=INTENSITY_COLOR,lw=1.7)
        peak_ax.set(ylabel='Peak I\n('+peakunit+')')
        for index,width in enumerate(width_arrays):
            width_ax.plot(result.z_m/zscale,width/widthscale,label='x' if index==0 else 'y',
                          color=WIDTH_COLORS[index],lw=2.3 if index==0 else 1.7,
                          ls='-' if index==0 else (0,(4,2.5)),alpha=.85 if index==0 else .95)
        shade_depth_of_focus(width_ax,result,zscale,WIDTH_COLORS)
        if len(width_arrays)>1:
            outside_legend(width_ax)
        width_label=(r'$\mathrm{FWHM}_x$' if len(width_arrays)==1 else r'$\mathrm{FWHM}_{x,y}$')
        width_ax.set(xlabel=axial_label(result,zunit),ylabel=width_label+' ('+widthunit+')')
        current_z=selected_plane.z_m
        for ax in (*map_axes,peak_ax,width_ax):
            fixed_axes(ax)
            ax.axvline(result.peak_z_m/zscale,color=palette['secondary'],ls=':',lw=.85)
            ax.axvline(current_z/zscale,color=CURSOR_COLOR,lw=1.05,
                         path_effects=[patheffects.Stroke(linewidth=2.0,foreground=palette['surface'],alpha=.45),patheffects.Normal()])
        peak_ax.plot([current_z/zscale],[selected_plane.metrics.max_intensity/peakscale],'o',color=CURSOR_COLOR,ms=4,mec=palette['foreground'],mew=.45)
        for width in np.atleast_1d(selected_plane.metrics.fwhm)[:len(width_arrays)]:
            if width is not None and np.isfinite(width):
                width_ax.plot([current_z/zscale],[width/widthscale],'o',color=CURSOR_COLOR,ms=4,mec=palette['foreground'],mew=.45)
        extrema=mark_scan_extrema(result,peak_ax,width_ax,width_arrays,zscale,peakscale,widthscale)
        for ax in (*map_axes,peak_ax):
            ax.tick_params(labelbottom=False)
        for ax in (peak_ax,width_ax):
            ax.grid(alpha=.5)
        state['scan_extrema']=extrema
        state['scan_axes']=(*map_axes,peak_ax,width_ax)
        set_render_value(scan_image,plot_png(figure,scan_compact_image,(3.6,8.4)))
        state['scan_cursor_m']=current_z
        state['scan_render_key']=(result,current_theme,current_z)


def nearest_z(z_m):
    """Snap physical z to the regular computed overview grid."""
    positions=state['result'].z_m
    return float(positions[np.argmin(abs(positions-float(z_m)))])


_MISSING_RENDER_STATE=object()
_RENDER_STATE_KEYS=('last_plane','plane_updates','scan_axes','scan_cursor_m',
    'scan_render_key','plane_axes','colorbar_axes','fwhm_annotations','map_fwhm_rulers',
    'map_fwhm_grouped','profile_peak','profile_fwhm_height','scan_extrema')


def _restore_render_state(snapshot):
    for key,value in snapshot.items():
        if value is _MISSING_RENDER_STATE:state.pop(key,None)
        else:state[key]=value


def _sync_navigation(index,result):
    state['syncing_navigation']=True
    try:
        state['slider'].value=index
        state['z_input'].value=z_text(result.z_m[index])
        state['previous_button'].disabled=index==0
        state['next_button'].disabled=index==len(result.z_m)-1
        state['plane_label'].value=f'<span style="font-size:12px">Plane {index+1} / {len(result.z_m)}</span>'
    finally:
        state['syncing_navigation']=False


def select_plane(index):
    """Render the latest request and commit one consistent plane/cursor frame.

    Reentrant input replaces the pending request instead of being dropped.
    Plot bytes and readouts are staged; a failed or superseded render cannot
    publish a partial frame or mark an old overview image as up to date.
    """
    result=state.get('result')
    if result is None:return
    index=int(np.clip(index,0,len(result.z_m)-1))
    state['pending_selection']=(result,index)
    if state.get('updating_selection'):return
    state['updating_selection']=True
    initial_result=result
    ready_progress=progress_view.value if state.get('progress_stage')=='ready' else None
    if ready_progress is not None:
        progress_view.value='<div style="display:flex;align-items:center;font-size:11px;margin:1px 0 3px"><span>Updating selected plane…</span></div>'
    try:
        while state.get('pending_selection') is not None:
            result,index=state['pending_selection']
            state['pending_selection']=None
            if state.get('result') is not result:continue
            snapshot={key:state.get(key,_MISSING_RENDER_STATE) for key in _RENDER_STATE_KEYS}
            frame={};state['render_values']=frame
            theme=current_theme
            try:
                _render_plane(float(result.z_m[index]))
                pending=state.get('pending_selection')
                if state.get('result') is not result:continue
                if (pending is not None and (pending[0] is not result or pending[1]!=index)) or theme!=current_theme:
                    _restore_render_state(snapshot)
                    if state.get('pending_selection') is None:state['pending_selection']=(result,index)
                    continue
                if pending is not None:state['pending_selection']=None
                # All expensive work has finished before controls/readouts change.
                navigation=[state[name] for name in ('slider','z_input','previous_button','next_button','plane_label')]
                with ExitStack() as stack:
                    for widget in (*frame,*navigation):stack.enter_context(widget.hold_sync())
                    for widget,value in frame.items():widget.value=value
                    _sync_navigation(index,result)
                    state['displayed_index']=index
                    state['frame_revision']=state.get('frame_revision',0)+1
                    state['error']=None
                if state.get('ready_status') and state.get('progress_stage')=='ready':status.value=state['ready_status']
            except Exception as exc:
                if state.get('result') is result:_restore_render_state(snapshot)
                if state.get('pending_selection') is not None:continue
                state['error']=exc
                previous=state.get('displayed_index')
                if previous is not None and state.get('result') is result:
                    _sync_navigation(previous,result)
                status.value='<span style="font-size:12px;color:#b45309">Plane update failed: '+html.escape(str(exc))+'</span>'
            finally:
                state.pop('render_values',None)
    finally:
        state['updating_selection']=False
        if ready_progress is not None and state.get('result') is initial_result and state.get('progress_stage')=='ready':
            progress_view.value=ready_progress


def draw_plane(z_m):
    """Use the same selection path for direct z requests and theme changes."""
    result=state.get('result')
    if result is not None:select_plane(int(np.argmin(abs(result.z_m-float(z_m)))))


def build_plane_controls(result,zscale,zunit):
    index=int(np.argmin(abs(result.z_m-result.peak_z_m)))
    slider=widgets.IntSlider(value=index,min=0,max=len(result.z_m)-1,step=1,
        description='',readout=False,continuous_update=True,behavior='drag-snap',
        layout=widgets.Layout(width='auto',flex='1 1 240px',min_width='180px'))
    z_input=widgets.Text(value=z_text(result.z_m[index]),
        description=f'z ({zunit})',continuous_update=False,
        style={'description_width':'initial'},layout=widgets.Layout(width='160px'))
    previous=widgets.Button(icon='chevron-left',tooltip='Previous z plane',layout=widgets.Layout(width='34px'))
    following=widgets.Button(icon='chevron-right',tooltip='Next z plane',layout=widgets.Layout(width='34px'))
    label=widgets.HTML(layout=widgets.Layout(width='105px'))
    slider.style.handle_color=CURSOR_COLOR
    peak_tooltip=('Go to the strongest evaluated plane' if result.numerical_peak.sampled
                  else 'Go to the numerical peak-intensity plane')
    reference=widgets.Button(description='Peak',icon='crosshairs',tooltip=peak_tooltip,layout=widgets.Layout(width='77px'))
    reference.add_class('xposim-peak-plane')
    slider.add_class('xposim-plane-slider')
    z_input.add_class('xposim-z-input')
    previous.add_class('xposim-previous-plane')
    following.add_class('xposim-next-plane')
    label.add_class('xposim-plane-label')
    state.update(slider=slider,z_input=z_input,previous_button=previous,
                 next_button=following,peak_button=reference,plane_label=label)
    def slider_changed(change):
        if state.get('syncing_navigation') or state.get('slider') is not slider or state.get('result') is not result:return
        select_plane(change['new'])
    def z_changed(change):
        if state.get('syncing_navigation') or state.get('z_input') is not z_input or state.get('result') is not result:return
        try:
            target=float(change['new'])*state['z_unit_scale']
            if not np.isfinite(target):
                raise ValueError('Enter a finite z value.')
            select_plane(int(np.argmin(abs(result.z_m-target))))
        except (ValueError,TypeError) as exc:
            status.value='<span style="font-size:12px;color:#b45309">'+html.escape(str(exc))+'</span>'
    slider.observe(slider_changed,names='value')
    z_input.observe(z_changed,names='value')
    def navigate(target):
        if state.get('slider') is slider and state.get('result') is result:select_plane(target)
    def step(delta):
        pending=state.get('pending_selection')
        base=pending[1] if pending is not None and pending[0] is result else slider.value
        navigate(base+delta)
    previous.on_click(lambda _:step(-1))
    following.on_click(lambda _:step(1))
    reference.on_click(lambda _:navigate(index))
    state['handler']=slider_changed
    plane_controls.children=(widgets.HBox([previous,slider,following,reference,z_input,label],
        layout=widgets.Layout(width='100%',align_items='center',flex_flow='row wrap')), )
    plane_controls.children[0].add_class('xposim-selection-row')
    select_plane(index)


def half_max_crossings(coordinate,values):
    values=np.asarray(values);coordinate=np.asarray(coordinate)
    i=int(np.argmax(values));half=values[i]/2
    left=np.flatnonzero(values[:i]<half)
    right=np.flatnonzero(values[i+1:]<half)+i+1
    if not left.size or not right.size or half<=0:return None
    l=int(left[-1]);r=int(right[0])
    def crossing(a,b):
        return float(coordinate[a]+(half-values[a])/(values[b]-values[a])*(coordinate[b]-coordinate[a]))
    return crossing(l,l+1),crossing(r-1,r)


def annotate_fwhm(ax,profiles,scale,axes,frame,*,style=None,fontsize=10):
    style=palette if style is None else style
    xmin,xmax=ax.get_xlim();ymin,ymax=ax.get_ylim()
    span=xmax-xmin
    measurements=[]
    for coord,profile,axis in profiles:
        if axis not in axes:continue
        crossing=half_max_crossings(coord,profile)
        label=r'$\mathrm{FWHM}_{'+axis+'}$: '
        if crossing is None:
            measurements.append((axis,None,label+'unresolved',0.0))
            continue
        lower,upper=np.asarray(crossing)/scale
        width=crossing[1]-crossing[0]
        unit_scale,unit=(1e-9,'nm') if width<1e-6 else (1e-6,'µm')
        label+=compact_number(width/unit_scale)+' '+unit
        axis_span=(ymax-ymin) if axis=='x' else span
        measurements.append((axis,crossing,label,(upper-lower)/axis_span))
    # When either width is too small, reserve one shared readout below the map.
    grouped=any(crossing is None or fraction<.05 for _,crossing,_,fraction in measurements)
    annotations=[];rulers=[]
    caption=dict(fontsize=fontsize,color=style['foreground'],
                 bbox=dict(facecolor=style['surface'],alpha=.9,edgecolor='none',pad=1.5))
    for axis,crossing,label,_ in measurements:
        if crossing is None:continue
        lower,upper=np.asarray(crossing)/scale
        if not grouped:
            color=style['blue'] if axis=='x' else style['other']
            if axis=='y':
                y=ymin+.2*(ymax-ymin)
                start,end=(lower,y),(upper,y)
                ax.text((lower+upper)/2,y-.06*span,label,ha='center',va='top',**caption)
            else:
                x=xmin+.18*span
                start,end=(x,lower),(x,upper)
                ax.text(x-.055*span,(lower+upper)/2,label,rotation=90,ha='right',va='center',**caption)
            ruler=ax.annotate('',xy=end,xytext=start,arrowprops=dict(
                arrowstyle='|-|',color=color,lw=1.1,shrinkA=0,shrinkB=0,mutation_scale=fontsize))
            ruler.set_gid('xposim-map-fwhm-ruler')
            rulers.append(axis)
        annotations.append((axis,tuple(map(float,crossing))))
    if grouped:
        ax.set_xlabel('\n'.join(filter(None,[ax.get_xlabel(),*(label for _,_,label,_ in measurements)])))
        ax.xaxis.label.set_linespacing(1.3)
        ax.xaxis.label.set_gid('xposim-fwhm-group')
    frame['fwhm_annotations']=annotations
    frame['map_fwhm_rulers']=rulers
    frame['map_fwhm_grouped']=grouped


def annotate_map_peak(ax,plane):
    ax.text(.97,.97,'Peak intensity\n'+plot_metric(plane.metrics.max_intensity,'a.u.'),
        transform=ax.transAxes,ha='right',va='top',fontsize=10,
        color=palette['foreground'],
        bbox=dict(facecolor=palette['surface'],edgecolor=palette['border'],alpha=.92,pad=3))


def plot_metric(value,unit=''):
    if value is None or not np.isfinite(value):return 'unresolved'
    exponent=int(np.floor(np.log10(abs(value)))) if value and (abs(value)<.1 or abs(value)>=1e4) else 0
    number=compact_number(value/10.0**exponent)
    if exponent:number=r'$'+number+r'\times 10^{'+str(exponent)+'}$'
    return number+((' '+unit) if unit else '')


def annotate_profile(ax,coordinate,values,scale,reference,frame,show_peak=True,*,style=None,show_values=True):
    style=palette if style is None else style
    index=int(np.argmax(values))
    peak=float(values[index])/reference
    label='Peak intensity\n'+plot_metric(float(values[index]),'a.u.')
    if show_peak:ax.plot([coordinate[index]/scale],[peak],'o',ms=4,color=style['blue'],zorder=5)
    if show_peak and show_values:ax.annotate(label,xy=(coordinate[index]/scale,peak),xytext=(.98,.93),
        textcoords='axes fraction',ha='right',va='top',fontsize=11,
        color=style['foreground'],arrowprops=dict(arrowstyle='-',color=style['blue'],lw=.9),
        bbox=dict(facecolor=style['surface'],edgecolor=style['border'],alpha=.95,pad=4))
    crossing=half_max_crossings(coordinate,values)
    frame['profile_peak']=(float(coordinate[index]),float(values[index])) if show_peak else None
    frame['fwhm_annotations']=[]
    if crossing is None:
        if show_values:ax.text(.5,.72,r'$\mathrm{FWHM}_x$: unresolved',transform=ax.transAxes,ha='center',
            color=style['foreground'],fontsize=11)
        return
    left,right=np.asarray(crossing)/scale
    height=peak/2
    ax.plot([left,right],[height,height],color=style['green'],lw=2.,
            solid_capstyle='butt',gid='xposim-fwhm-segment',zorder=5)
    width=crossing[1]-crossing[0]
    unit_scale,unit=(1e-9,'nm') if width<1e-6 else (1e-6,'µm')
    label=r'$\mathrm{FWHM}_x$ = '+compact_number(width/unit_scale)+' '+unit
    if show_values:ax.annotate(label,xy=((left+right)/2,height),xytext=(0,6),textcoords='offset points',
        ha='center',va='bottom',fontsize=11,color=style['foreground'],
        bbox=dict(facecolor=style['surface'],alpha=.9,edgecolor='none',pad=2))
    frame['fwhm_annotations']=[('x',tuple(map(float,crossing)))]
    frame['profile_fwhm_height']=height


def transverse_sample_bounds(result,plane):
    return tuple((float(axis[0]-step/2),float(axis[-1]+step/2))
                 for axis,step in ((plane.x_m,result.detector_step_m),(plane.y_m,result.detector_step_y_m)))


def draw_transverse_map(ax,result,plane,reference,view):
    """Physical bounds, normalized image and peak-centred cuts."""
    x_bounds,y_bounds=view if view is not None else transverse_sample_bounds(result,plane)
    scale,unit=length_units([*x_bounds,*y_bounds])
    j,i=np.unravel_index(int(np.argmax(plane.intensity)),plane.intensity.shape)
    profiles=[(plane.x_m,plane.intensity[j,:],'x'),(plane.y_m,plane.intensity[:,i],'y')]
    mesh=ax.pcolormesh(plane.y_m/scale,plane.x_m/scale,plane.intensity.T/reference,
        shading='auto',cmap=CMAP,norm=Normalize(0,1),rasterized=True)
    ax.set(xlabel=f'y ({unit})',ylabel=f'x ({unit})',xlim=np.asarray(y_bounds)/scale,
           ylim=np.asarray(x_bounds)/scale,aspect='equal',adjustable='box')
    return mesh,profiles,scale


def draw_transverse_profiles(ax,result,profiles,reference,view,*,style=None,line_widths=(2.3,1.7)):
    style=palette if style is None else style
    scale,unit=length_units(np.concatenate([coord for coord,_,_ in profiles]))
    for index,(coord,profile,axis) in enumerate(profiles):
        ax.plot(coord/scale,profile/reference,label=axis,
                color=style['blue'] if index==0 else style['other'],lw=line_widths[index],
                ls=(0,(4,2.5)) if profile_is_dashed(axis,result) else '-',
                alpha=.85 if index==0 else .95,zorder=2+index)
    ax.set(xlabel=f'Transverse coordinate ({unit})' if len(profiles)>1 else f'x ({unit})',
           ylabel='I / Iref',ylim=(-.035,1.05))
    if view is not None:
        limits=view[:len(profiles)]
        ax.set_xlim(min(lo for lo,_ in limits)/scale,max(hi for _,hi in limits)/scale)
    return scale,unit


def render_transverse(result,plane):
    frame=state
    peak=max(result.intensity_reference,np.finfo(float).tiny)
    view=frame.get('transverse_view')
    with plt.rc_context(plot_style()):
        figure=plt.figure(figsize=PLANE_SIZE,layout='constrained')
        if plane.y_m is not None:
            bounds=view if view is not None else transverse_sample_bounds(result,plane)
            view_aspect=float(np.ptp(bounds[0])/np.ptp(bounds[1]))
            thin=min(view_aspect,1/view_aspect)<.25
            map_height=max(.9,min(3.2,3.2*view_aspect+.3))
            spec=figure.add_gridspec(2,1,height_ratios=[map_height,1.2],hspace=.08)
            map_ax=figure.add_subplot(spec[0,0]);profile_ax=figure.add_subplot(spec[1,0])
            mesh,profiles,scale=draw_transverse_map(map_ax,result,plane,peak,bounds)
            map_ax.set_title('Transverse intensity · shared ROI' if view is not None else 'Transverse intensity')
            fixed_axes(map_ax);map_ax.set_facecolor(CMAP(0))
            map_ax.yaxis.tick_right();map_ax.yaxis.set_label_position('right')
            label_narrow_map_axis(map_ax)
            if thin:
                map_ax.set_title('Transverse intensity\nPeak intensity '+plot_metric(plane.metrics.max_intensity,'a.u.'),
                                 fontsize=11)
                cb=figure.colorbar(mesh,ax=map_ax,location='top',aspect=28,shrink=.54,pad=.08,
                                   ticks=[0,.5,1],label='I / Iref')
                cb.ax.xaxis.set_major_formatter(TICK_FORMATTER)
                cax=cb.ax
            else:
                cax=map_ax.inset_axes([-.13,0,.035,1])
                cb=figure.colorbar(mesh,cax=cax);cb.set_ticks([0,.5,1])
                cb.ax.yaxis.set_ticks_position('left');cb.ax.yaxis.set_major_formatter(TICK_FORMATTER)
                cb.ax.set_ylabel('I / Iref',fontsize=10,color=palette['foreground'])
                cb.ax.yaxis.set_label_position('right')
            cb.ax.tick_params(labelsize=9,colors=palette['secondary'])
            cb.outline.set_visible(False)
            frame['fwhm_annotations']=[];frame['map_fwhm_rulers']=[];frame['map_fwhm_grouped']=False
            if len(result.fwhm_axes)==2:
                annotate_fwhm(map_ax,profiles,scale,result.fwhm_axes,frame)
            else:
                profiles=profiles[:1]
            if not thin:annotate_map_peak(map_ax,plane)
            frame['plane_axes']=(map_ax,profile_ax);frame['colorbar_axes']=cax
            view=bounds
        else:
            profile_ax=figure.add_subplot(111)
            profiles=[(plane.x_m,plane.intensity,'x')]
            frame['plane_axes']=(profile_ax,);frame['fwhm_annotations']=[]
        scale,_=draw_transverse_profiles(profile_ax,result,profiles,peak,view)
        profile_ax.set_title('' if len(profiles)>1 else ('Focused x profile' if plane.y_m is not None else 'Transverse intensity'))
        if len(profiles)>1:outside_legend(profile_ax)
        profile_ax.grid(alpha=.5);fixed_axes(profile_ax)
        if len(profiles)==1:annotate_profile(profile_ax,profiles[0][0],profiles[0][1],scale,peak,frame,show_peak=plane.y_m is None)
        set_render_value(plane_image,plot_png(figure,plane_compact_image,(3.6,5.0) if plane.y_m is not None else (3.6,4.2)))


def _render_plane(z_m):
    result=state['result']
    if result is None:return
    plane=result.plane_at(nearest_z(z_m))
    state['last_plane']=plane;state['plane_updates']+=1
    render_transverse(result,plane)
    scan_key=state.get('scan_render_key')
    if scan_key is None or scan_key[0] is not result or scan_key[1]!=current_theme or scan_key[2]!=plane.z_m:
        render_scan(result,plane)
    update_summary(result,plane)


STAGE_LABELS={'preparing':'Preparing calculation','grid':'Preparing CRL and grid',
    'source_preparation':'Preparing the full Gaussian source',
    'window_checks':'Checking propagation window','source_modes':'Preparing source modes',
    'detector':'Preparing transverse evaluation grid','peak_search':'Locating the peak-intensity plane',
    'axial_roi':'Finding display interval','sampling':'Calculating z planes',
    'finalizing':'Finalizing beam metrics','equal_fwhm':'Locating the equal-FWHM plane',
    'complete':'Numerical calculation complete',
    'rendering':'Rendering plots','ready':'Complete','failed':'Calculation stopped'}


def duration_text(seconds):
    seconds=max(0,int(np.ceil(seconds)))
    minutes,seconds=divmod(seconds,60)
    hours,minutes=divmod(minutes,60)
    return (f'{hours:d}:{minutes:02d}:{seconds:02d}' if hours else f'{minutes:02d}:{seconds:02d}')


def show_progress(stage,completed=0,total=None,elapsed_s=0.0,eta_s=None,detail=''):
    label=STAGE_LABELS.get(stage,stage)
    if stage=='ready':
        progress_view.value='<div style="display:flex;align-items:center;gap:8px;font-size:11px;opacity:.8;margin:1px 0 3px"><progress max="1" value="1" aria-label="Complete" style="width:85px;height:8px;accent-color:#3d7665"></progress><span>Complete · '+duration_text(elapsed_s)+'</span></div>'
        progress_view.layout.display=''
        state['progress_stage']=stage
        return
    if total is None:
        amount=f'{completed} evaluations' if stage in {'peak_search','axial_roi','equal_fwhm'} else ''
        bar='<progress aria-label="'+html.escape(label,quote=True)+'" style="width:100%;height:12px;accent-color:#355f85"></progress>'
    else:
        amount=f'{completed} / {total}'
        bar='<progress aria-label="'+html.escape(label,quote=True)+'" max="'+str(total)+'" value="'+str(completed)+'" style="width:100%;height:12px;accent-color:#355f85"></progress>'
    if stage=='ready':
        remaining='Completed'
    elif stage=='failed':
        remaining='Stopped'
    elif eta_s is not None:
        remaining='ETA for this stage: ≈ '+duration_text(eta_s)
    elif total is None:
        remaining='ETA: estimating · adaptive stage' if stage in {'grid','peak_search','axial_roi','equal_fwhm'} else 'ETA: estimating'
    else:
        remaining='ETA for this stage: estimating'
    progress_view.value='<div style="padding:7px 10px;margin:3px 0 5px;border:1px solid rgba(127,127,127,.2);border-radius:8px;font-size:12px"><div style="display:flex;justify-content:space-between;gap:12px"><b>'+html.escape(label)+'</b><span>'+html.escape(amount)+'</span></div>'+bar+'<div style="display:flex;justify-content:space-between;gap:12px;flex-wrap:wrap"><span>Elapsed: '+duration_text(elapsed_s)+'</span><span>'+html.escape(remaining)+'</span></div>'+('<div style="font-size:11px;opacity:.75;margin-top:3px">'+html.escape(detail)+'</div>' if detail else '')+'</div>'
    progress_view.layout.display=''
    state['progress_stage']=stage


def on_calculation_progress(update):
    state['last_progress']=update
    show_progress(update.stage,update.completed,update.total,update.elapsed_s,update.eta_s,update.detail)


def compute_focus(_=None):
    started=time.perf_counter()
    apply_theme(refresh=False)
    show_progress('preparing')
    compute_button.disabled=True
    report_button.disabled=True;report_message.value=''
    status.value='<span style="font-size:12px"><progress style="width:90px;height:8px"></progress> Computing…</span>'
    state['error']=None
    for name in ('slider','z_input','previous_button','next_button','peak_button','plane_label'):
        widget=state.get(name)
        if widget is not None:
            widget.unobserve_all();widget.close()
        state[name]=None
    state['slider']=None;state['result']=None;state['pending_selection']=None
    state.pop('last_plane',None)
    state.pop('scan_cursor_m',None)
    state.pop('scan_render_key',None)
    state.pop('displayed_index',None)
    summary.value='';scan_image.value=b'';plane_image.value=b'';scan_compact_image.value=b'';plane_compact_image.value=b'';plane_metrics.value='';plane_heading.value=''
    results_panel.layout.display='none';details.layout.display='none';plane_controls.layout.display='none'
    source_comparison.value=''
    source_comparison.layout.display='none'
    gc.collect()
    try:
        config=read_configuration()
        result=xpos.estimate_crl_focus(config,progress=on_calculation_progress)
        elapsed=time.perf_counter()-started
        state['result']=result
        state['analytical_result']=result.crossed_alignment or xpos.analyze_crl(result.configuration)
        state['transverse_view']=result.display_signal_bounds_m
        plane_compact_image.height='588' if result.configuration.grid_dimension==1 else '700'
        state['intensity_reference']=max(result.intensity_reference,np.finfo(float).tiny)
        show_progress('rendering',elapsed_s=time.perf_counter()-started)
        zscale,zunit=axial_units(result)
        state['z_unit_scale']=zscale;state['z_unit']=zunit
        build_plane_controls(result,zscale,zunit)
        if state['error'] is not None:
            raise state['error']
        results_panel.layout.display='flex'
        plane_controls.layout.display=''
        details.layout.display=''
        source_comparison.value=source_width_comparison(result)
        source_comparison.layout.display='' if source_comparison.value else 'none'
        source_label=('Plane wave' if config.source_type=='plane' else 'Point source' if config.source_type=='point' else 'Incoherent Gaussian source · FWHM '+axis_values(
            (config.source_fwhm_x_m,) if config.grid_dimension==1 else (config.source_fwhm_x_m,config.source_fwhm_y_m),width_value))
        alignment=result.crossed_alignment
        if alignment is None:
            material_label=('Direct δ / β' if config.material_delta is not None else html.escape(config.material_formula)+' · '+metric_value(config.density_g_cm3,'g/cm³'))
            lens_rows=[('Material',material_label),('δ',metric_value(result.delta)),('β',metric_value(result.beta)),
                       ('Radius of curvature / geometric aperture',distance_value(config.radius_m)+' / '+distance_value(config.aperture_m)),
                       ('CRL length',distance_value(result.lens_length_m)),
                       ('Equivalent aperture from transmission',distance_value(result.effective_aperture_m))]
        else:
            lens_rows=[('x CRL',html.escape(schematic_material(config))),('y CRL',html.escape(schematic_material(config.second_crl))),
                       ('Inter-stack gap',alignment_length(alignment.gap_m)),
                       ('x image · from y exit',alignment_length(alignment.image_x_m)),
                       ('y image · from y exit',alignment_length(alignment.image_y_m))]
        rows=[('Source',source_label),
              *([] if config.source_type=='plane' else [('Source-to-CRL entrance',distance_value(config.source_distance_m))]),
              *([('Projected source FWHM at reference',axis_values(result.projected_source_fwhm_at_peak_m,width_value))]
                if config.source_type=='gaussian' else []),
              *([('Source point counts',axis_values(result.source_mode_counts,str))]
                if result.source_mode_counts is not None else []),
              *lens_rows,
              ('Propagation grid',str(result.grid_shape)),
              ('Transverse evaluation grid',str(result.detector_shape)),
              ('Propagation x step',distance_value(result.grid_step_m)),
              ('Evaluation x step',distance_value(result.detector_step_m)),
              ('Peak search illumination','Plane wave' if config.source_type=='plane' else
               'Full Gaussian source' if result.peak_search_basis=='gaussian_source' else
               'Central source point' if config.source_type=='gaussian' else 'Point source'),
              ('Peak search evaluations',str(result.numerical_peak.evaluations)),
              *([('Central-point peak used for the scan estimate',distance_value(result.coherent_reference.z_m))]
                if result.peak_search_basis=='gaussian_source' else
                [('Peak position for the Gaussian source (sampled)',distance_value(result.sampled_peak_z_m))]
                if config.source_type=='gaussian' else [])]
        detail='<div style="font-size:12px;line-height:1.6">'+''.join('<div><b>'+label+':</b> '+value+'</div>' for label,value in rows)
        diagnostic_text=(message.partition(':')[2].strip().replace('detector ROI','calculation region')
                         .replace('detector sampling','transverse evaluation sampling') for message in result.diagnostics)
        detail+=''.join('<div>'+html.escape(message)+'</div>' for message in diagnostic_text)
        details_text.value=detail+'</div>'
        details.set_title(0,f'Model, mesh and diagnostics ({len(result.diagnostics)})')
        if result.backend=='jax':
            device='JAX · '+result._session.grid.x.device.device_kind
        elif result.backend=='torch':
            import torch
            device='PyTorch / CUDA · '+torch.cuda.get_device_name(result._session.grid.x.device)
        else:
            device='CPU / NumPy' if result.backend=='numpy' else 'CuPy / '+_cuda_device_name()
        elapsed=time.perf_counter()-started
        status.value=runtime_grid_info(result,device,elapsed)
        state['ready_status']=status.value
        state['report_runtime']=(device,elapsed)
        report_button.disabled=False
        report_button.tooltip=('Export a one-page summary with sections before, at and after the image plane.' if alignment is not None else
                               'Export a one-page summary with transverse sections around the peak reference.')
        show_progress('ready',1,1,time.perf_counter()-started,0.0)
        sync_results_notice()
    except Exception as exc:
        show_progress('failed',elapsed_s=time.perf_counter()-started,detail=str(exc))
        state['error']=exc;state['result']=None
        summary.value='';scan_image.value=b'';plane_image.value=b'';scan_compact_image.value=b'';plane_compact_image.value=b'';plane_metrics.value='';plane_heading.value=''
        plane_controls.children=()
        source_comparison.value='';source_comparison.layout.display='none'
        results_panel.layout.display='none';details.layout.display='none';plane_controls.layout.display='none'
        status.value='<span style="font-size:12px;color:#b45309">'+html.escape(str(exc))+'</span>'
    finally:
        compute_button.disabled=False
        if state['result'] is None:
            report_button.disabled=True
            report_button.tooltip='Complete a calculation to export a report.'


def runtime_grid_info(result,device,elapsed):
    def grid_item(label,shape,dx,dy):
        count=str(shape[-1]) if len(shape)==1 else str(shape[-1])+' × '+str(shape[0])
        text=label+': '+count+' · Δx '+width_value(dx)
        if dy is not None:text+=' · Δy '+width_value(dy)
        return '<span style="white-space:normal">'+text+'</span>'
    items=['<b>'+html.escape(device)+'</b>','<span>'+compact_number(elapsed)+' s</span>',
           grid_item('Propagation grid',result.grid_shape,result.grid_step_m,result.grid_step_y_m)]
    if result.detector_shape!=result.grid_shape or result.detector_step_m!=result.grid_step_m or result.detector_step_y_m!=result.grid_step_y_m:
        items.append(grid_item('Transverse evaluation grid',result.detector_shape,result.detector_step_m,result.detector_step_y_m))
    items.append('<span>Axial scan: '+str(len(result.z_m))+' planes · Δz '+axial_interval(result.scan_step_m)+'</span>')
    return '<div style="font-size:13px;line-height:1.4;display:flex;flex-wrap:wrap;gap:3px 16px;align-items:baseline">'+''.join(items)+'</div>'


def _hardware_help():
    backend=xpos.select_backend('auto')
    if backend in ('torch','jax'):
        return 'Auto selects <b>'+{'torch':'PyTorch','jax':'JAX'}[backend]+' on GPU</b>.'
    if backend=='cupy':
        return 'Auto selects CuPy on <b>'+html.escape(_cuda_device_name())+'</b>.'
    return 'Auto selects <b>CPU / NumPy</b>. To enable GPU in Colab: <b>Runtime → Change runtime type → GPU → Save</b>, then reconnect and run all cells. The actual CUDA device is checked.'


def _cuda_device_name():
    try:
        device=xpos.cp.cuda.Device()
        name=xpos.cp.cuda.runtime.getDeviceProperties(device.id)['name']
        if isinstance(name,bytes):name=name.decode()
        return 'CUDA GPU · '+str(name)
    except Exception:
        return 'CuPy / CUDA GPU'

def sync_results_notice():
    result=state.get('result')
    if result is None:
        results_notice.layout.display='none'
        return
    try:changed=read_configuration()!=result.configuration
    except (ValueError,TypeError):changed=True
    results_notice.value='<div class="xposim-notice">Parameters changed. Press Compute to update these results.</div>' if changed else ''
    results_notice.layout.display='' if changed else 'none'


REPORT_PAGE=(210/25.4,297/25.4)
REPORT_MARKERS=('#355f85','#147a70','#9b592d')


def report_number(value,scale,unit):
    if value is None or not math.isfinite(value):
        return '—'
    scaled=value/scale
    return compact_number(scaled) if unit=='mm' else f'{scaled:.6g}'


def report_value(value,scale=1,unit=''):
    number=report_number(value,scale,unit)
    return number+(' '+unit if unit and number!='—' else '')


def report_values(values,scale=1,unit=''):
    text=[report_number(value,scale,unit) for value in np.atleast_1d(values)]
    if len(text)==1 or text[0]==text[1]:
        combined=text[0]
    else:
        combined='x '+text[0]+' / y '+text[1]
    return combined+(' '+unit if unit and combined!='—' else '')


def report_depth_of_focus(result):
    widths=result.depth_of_focus_m
    smallest=min((width for width in widths if width is not None),default=1e-3)
    return report_values(widths,*interval_units(smallest))


def report_planes(result,center=None,*,full=False):
    """Take a representative sampled plane on each side of an exact reference."""
    center=result.peak_z_m if center is None else center
    z=result.z_m
    atol=8*np.spacing(center)
    before=z[z<center-atol]
    after=z[z>center+atol]
    reaches=[abs(float(samples[i])-center) for samples,i in ((before,0),(after,-1)) if samples.size]
    widths=[value for value in result.depth_of_focus_m if value is not None]
    offset=min(max(widths)*.5 if widths else float(np.ptp(z))*.25,min(reaches,default=0))
    def sample(options,target):
        return result.plane_at(float(options[np.argmin(abs(options-target))]),full=full) if options.size else None
    if full or center!=result.peak_z_m:
        reference=result.plane_at(center,full=full)
    else:
        reference=xpos.CRLPlaneResult(center,result.best_intensity,result.x_m,result.y_m,
                                      result.reference_metrics,result.magnification_at_peak)
    return sample(before,center-offset),reference,sample(after,center+offset)


def report_axis(ax,*,grid=True):
    ax.tick_params(labelsize=7,length=2.5)
    for spine in ax.spines.values():
        spine.set_color('#a7b7c8')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    ax.xaxis.set_major_formatter(TICK_FORMATTER)
    ax.yaxis.set_major_formatter(TICK_FORMATTER)
    if grid:ax.grid(color='#dbe4ee',linewidth=.55)
    ax.set_axisbelow(True)


def report_scan(fig,result,planes,reference):
    """Longitudinal intensity sections and two trends on the same axial interval."""
    sections=longitudinal_sections(result)
    both=len(sections)==2
    map_boxes=([.104,.512,.830,.042],[.104,.469,.830,.036]) if both else ([.104,.494,.830,.061],)
    map_axes=[fig.add_axes(box) for box in map_boxes]
    trend_height=.061 if both else .080
    peak_ax=fig.add_axes([.104,.382,.351,trend_height])
    width_ax=fig.add_axes([.586,.382,.348,trend_height])
    zscale,zunit=axial_units(result)
    z=result.z_m/zscale
    for ax,section in zip(map_axes,sections):
        draw_longitudinal_section(ax,section,z,reference)
        ax.yaxis.label.set_size(8)
        if both:
            ax.text(.012,.85,f'{section[0]}–z',transform=ax.transAxes,color='#fff',
                    fontsize=7.5,va='top',bbox=dict(facecolor='#263d56',edgecolor='none',alpha=.82,pad=1.5))
    peak_scale,peak_unit=engineering(result.peak_intensity,'a.u.')
    peak_ax.plot(z,result.peak_intensity/peak_scale,color='#355f85',lw=1.35)
    peak_ax.set_title('Peak intensity',loc='left',fontsize=8.5,pad=4)
    peak_ax.set_ylabel(peak_unit,fontsize=7.2)
    widths=np.reshape(result.fwhm_m,(z.size,-1))
    width_colors=('#356a57','#759c8c')
    width_arrays=[widths[:,i] for i in range(len(result.fwhm_axes))]
    width_scale,width_unit=length_units(np.concatenate(width_arrays))
    for i,(axis,values) in enumerate(zip(result.fwhm_axes,width_arrays)):
        width_ax.plot(z,values/width_scale,color=width_colors[i],lw=1.45,ls='--' if i else '-',label=axis)
    shade_depth_of_focus(width_ax,result,zscale,width_colors)
    if len(result.fwhm_axes)>1:
        width_ax.legend(loc='upper right',bbox_to_anchor=(1,1.32),fontsize=6.8,ncol=2,
                        frameon=False,handlelength=1.35,borderaxespad=0)
    width_ax.set_title('FWHM',loc='left',fontsize=8.5,pad=4)
    width_ax.set_ylabel(width_unit,fontsize=7.4)
    for ax in (*map_axes,peak_ax,width_ax):
        is_map=ax in map_axes
        report_axis(ax,grid=not is_map)
        if is_map:
            ax.tick_params(labelbottom=False)
            if both:
                ax.yaxis.set_major_locator(MaxNLocator(nbins=2))
        pad=.5*result.scan_step_m/zscale if z.size==1 else 0.
        ax.set_xlim(z[0]-pad,z[-1]+pad)
        for plane,color in zip(planes,REPORT_MARKERS):
            if plane is not None:
                ax.axvline(plane.z_m/zscale,color='white' if is_map else color,
                           lw=.85,ls='-' if plane is planes[1] else ':',alpha=.9)
    mark_scan_extrema(result,peak_ax,width_ax,width_arrays,zscale,peak_scale,width_scale,printed=True)
    fig.text(.518,.356,axial_label(result,zunit),ha='center',fontsize=8,color='#394d64')


def report_transverse(fig,result,planes,reference,view):
    light=THEMES['light']
    two_dimensional=result.y_m is not None
    circular=result.fwhm_axes==('x','y')
    pane_x=(.088,.374,.660)
    panel_width=.243
    first_visible=next(i for i,plane in enumerate(planes) if plane is not None)
    zscale,zunit=axial_units(result)
    for i,(x,plane,label,color) in enumerate(zip(pane_x,planes,('A · Upstream','B · Reference plane','C · Downstream'),REPORT_MARKERS)):
        fig.text(x,.302,label,fontsize=8.6,weight='bold',color=color)
        bottom=(.215 if circular else .211) if two_dimensional else .151
        ax=fig.add_axes([x,bottom,panel_width,.275-bottom])
        if plane is None:
            ax.set_axis_off()
            ax.text(.5,.5,'Outside the sampled\nz range',ha='center',va='center',transform=ax.transAxes,
                    color='#526379',fontsize=8.5)
            continue
        show_y=i==first_visible
        if i!=1:
            fig.text(x,.283,f'z = {compact_number(plane.z_m/zscale)} {zunit}',fontsize=8,color='#435970')
        frame={}
        if two_dimensional:
            _,profiles,scale=draw_transverse_map(ax,result,plane,reference,view)
            report_axis(ax,grid=False)
            label_narrow_map_axis(ax)
            ax.set_facecolor(CMAP(0))
            # Units are in the first axis label of each set.
            if show_y:ax.yaxis.label.set_fontsize(6.5)
            else:
                ax.set_ylabel('')
                ax.tick_params(labelleft=False)
            ax.xaxis.label.set_fontsize(6.45)
            ax.xaxis.labelpad=1
            if circular:
                annotate_fwhm(ax,profiles,scale,result.fwhm_axes,frame,style=light,fontsize=6.2)
            else:
                profiles=profiles[:1]
            profile_ax=fig.add_axes([x,.126 if circular else .135,panel_width,.037])
        else:
            profiles=[(plane.x_m,plane.intensity,'x')]
            profile_ax=ax

        scale,unit=draw_transverse_profiles(profile_ax,result,profiles,reference,view,
                                            style=light,line_widths=(1.35,1.15))
        report_axis(profile_ax)
        profile_ax.yaxis.set_major_locator(MaxNLocator(nbins=2))
        if show_y:profile_ax.yaxis.label.set_fontsize(6.5)
        else:
            profile_ax.set_ylabel('')
            profile_ax.tick_params(labelleft=False)
        if two_dimensional:
            profile_ax.set_xlabel('')
            if show_y:
                caption=f'Profiles ({unit}) · x solid, y dashed' if circular else f'Focused x profiles ({unit})'
                fig.text(.08,.319,caption,fontsize=7,color='#526379')
        else:
            profile_ax.xaxis.label.set_fontsize(6.65)
            profile_ax.xaxis.labelpad=1
        if not circular:
            annotate_profile(profile_ax,profiles[0][0],profiles[0][1],scale,reference,frame,
                             show_peak=not two_dimensional,style=light,show_values=False)
            width=report_values(np.atleast_1d(plane.metrics.fwhm)[:1],1e-9,'nm')
            fig.text(x,.113,'FWHM  '+width,fontsize=7.5,color='#283b50')
        fig.text(x,.098,'Peak I  '+report_value(plane.metrics.max_intensity,unit='a.u.'),
                 fontsize=7.5,color='#283b50')
    if view is not None:fig.text(.403,.319,'Shared signal ROI',fontsize=7,color='#526379')


def crossed_report_planes(result):
    """Use the common paraxial image in the scan, or its numerical reference."""
    image=result.crossed_alignment.common_image_m
    bounds=result.configuration.scan_range_m or (result.z_m[0],result.z_m[-1])
    centered_on_image=(image is not None and bounds[0]<=image<=bounds[1]
                       and result.z_m[0]<=image<=result.z_m[-1])
    return report_planes(result,image if centered_on_image else None,full=True),centered_on_image


def crossed_report_view(planes,result):
    """Fit independent physical bounds around the beam cores of all sections."""
    active=[plane for plane in planes if plane is not None]
    available=transverse_sample_bounds(result,active[0])
    bounds=[]
    for dimension,axis,step in ((0,'x_m',result.detector_step_m),(1,'y_m',result.detector_step_y_m)):
        intervals=[]
        for plane in active:
            coordinates=getattr(plane,axis)
            projection=np.max(plane.intensity,axis=dimension)
            peak=int(np.argmax(projection))
            if projection[peak]<=0:
                continue
            threshold=result.configuration.transverse_roi_level*projection[peak]
            left=np.flatnonzero(projection[:peak]<=threshold)
            right=np.flatnonzero(projection[peak+1:]<=threshold)
            low=int(left[-1])+1 if left.size else 0
            high=peak+int(right[0]) if right.size else len(coordinates)-1
            intervals.append((coordinates[low]-step/2,coordinates[high]+step/2))
        lo,hi=available[dimension]
        if intervals:
            low=min(left for left,_ in intervals)
            high=max(right for _,right in intervals)
            margin=.07*(high-low)
            lo,hi=max(lo,low-margin),min(hi,high+margin)
        bounds.append((lo,hi))
    return tuple(bounds)


def report_crossed_transverse(fig,result,planes,reference,centered_on_image):
    light=THEMES['light']
    view=crossed_report_view(planes,result)
    zscale,zunit=axial_units(result)
    captions=(('A · Before focus','B · Image plane','C · After focus') if centered_on_image
              else ('A · Upstream','B · Numerical reference','C · Downstream'))
    panel_width=.239
    fig.text(.301,.330,'Profiles · x solid, y dashed',fontsize=7.1,color='#526379')
    for i,(left,plane,caption,color) in enumerate(zip((.091,.385,.679),planes,captions,REPORT_MARKERS)):
        fig.text(left,.308,caption,fontsize=8.4,weight='bold',color=color)
        map_ax=fig.add_axes([left,.215,panel_width,.078])
        if plane is None:
            map_ax.set_axis_off()
            map_ax.text(.5,.65,'Outside the sampled\nz range',ha='center',va='center',
                        transform=map_ax.transAxes,fontsize=8,color='#526379')
            continue

        fig.text(left,.295,f'z = {compact_number(plane.z_m/zscale)} {zunit}',
                 fontsize=7.4,color='#43576e')

        def section_slice(axis,bounds):
            before,after=axis.searchsorted(bounds)
            return slice(max(0,before-1),min(len(axis),after+1))

        xs,ys=section_slice(plane.x_m,view[0]),section_slice(plane.y_m,view[1])
        visible=xpos.CRLPlaneResult(plane.z_m,plane.intensity[ys,xs],plane.x_m[xs],plane.y_m[ys],
                                   plane.metrics,plane.magnification)
        _,profiles,_=draw_transverse_map(map_ax,result,visible,reference,view)
        report_axis(map_ax,grid=False)
        label_narrow_map_axis(map_ax)
        map_ax.set_facecolor('#edf2f7')
        map_ax.xaxis.label.set_size(7)
        map_ax.yaxis.label.set_size(7)
        map_ax.xaxis.labelpad=1
        map_ax.yaxis.labelpad=1

        wx,wy=plane.metrics.fwhm if plane.metrics.fwhm is not None else (None,None)
        def width_text(value):
            if value is None or not math.isfinite(value):
                return '—'
            scale,unit=(1e-9,'nm') if value<1e-6 else (1e-6,'µm')
            return compact_number(value/scale)+' '+unit

        width_labels=tuple(map(width_text,(wx,wy)))
        fig.text(left,.175,r'$\mathrm{FWHM}_{x}$  '+width_labels[0],fontsize=7.4,color='#43576e')
        fig.text(left+.126,.175,r'$\mathrm{FWHM}_{y}$  '+width_labels[1],fontsize=7.4,color='#43576e')
        fig.text(left,.159,'Peak I  '+report_value(plane.metrics.max_intensity,unit='a.u.'),
                 fontsize=7.4,color='#43576e')

        profile_ax=fig.add_axes([left,.116,panel_width,.032])
        _,unit=draw_transverse_profiles(profile_ax,result,profiles,reference,view,
                                        style=light,line_widths=(1.35,1.15))
        profile_ax.set_xlabel('')
        report_axis(profile_ax)
        if i==0 or planes[0] is None and i==1:
            profile_ax.set_ylabel('I / Iref',fontsize=6.5)
        else:
            profile_ax.set_ylabel('')
            profile_ax.tick_params(labelleft=False)
        profile_ax.set_yticks((0,1))
    fig.text(.518,.090,f'Transverse coordinate ({unit})',ha='center',fontsize=6.9,color='#394d64')


def report_measurements(fig,result,analytical):
    """Summarize ray-optical predictions and numerical results."""
    cfg=result.configuration
    magnification=analytical.magnification_at_image[:len(result.fwhm_axes)]
    predicted=(
        ('Back focal distance',report_value(analytical.collimated_focus_m,1e-3,'mm')),
        *(() if cfg.source_type=='plane' else (
            ('Image distance (analytical)',report_value(analytical.image_distance_m,1e-3,'mm')),
            ('Transverse magnification',report_values(magnification)),
        )),
        ('Effective aperture','∞' if math.isinf(analytical.kohn_effective_aperture_m)
         else report_value(analytical.kohn_effective_aperture_m,1e-6,'µm')),
        ('Rocking curve FWHM',report_value(analytical.rocking_fwhm_rad,1e-3,'mrad')),
        ('Web transmission',report_value(analytical.web_transmission,.01,'%')),
        ('δ / β',report_value(analytical.delta)+' / '+report_value(analytical.beta)),
    )
    measured=[
        ('Peak position (numerical)',report_value(result.peak_z_m,1e-3,'mm')),
        ('Depth of focus',report_depth_of_focus(result)),
        ('Transmission',report_value(result.transmission,.01,'%')),
    ]
    if cfg.source_type=='gaussian':
        lengths=analytical.transverse_coherence_length_m(result.peak_z_m)
        measured.extend((
            ('Coherence length at B',coherence_values(lengths,
                lambda value:'∞' if math.isinf(value) else report_value(value,1e-6,'µm'))),
            ('FWHM ratio (Gaussian/point)',report_values(result.source_broadening_factor)),
        ))
    report_metric_columns(fig,predicted,measured,'Analytical predictions')


def report_metric_columns(fig,predicted,measured,analytical_title):
    for rows,title,left,right in ((predicted,analytical_title,.062,.449),
                                  (measured,'Beam and transmission',.54,.938)):
        fig.text(left,.718,title,fontsize=9.6,weight='bold',color='#172b42')
        for i,(label,value) in enumerate(rows):
            y=.696-.0165*i
            fig.text(left,y,label,fontsize=7.7,color='#43576e')
            fig.text(right,y,value,fontsize=7.8,ha='right',color='#13273e',weight='semibold')


def report_color_scale(fig):
    fig.text(.615,.334,'I / Iref',fontsize=7.3,color='#526379',ha='right',va='center')
    color_axes=fig.add_axes([.633,.331,.301,.007])
    colorbar=ColorbarBase(color_axes,cmap=CMAP,norm=Normalize(0,1),orientation='horizontal',ticks=(0,.5,1))
    colorbar.ax.tick_params(labelsize=7,length=2,pad=1)
    colorbar.outline.set_visible(False)


def report_crossed_setup(fig,result,alignment):
    cfg=result.configuration
    y=cfg.second_crl
    geometry=[
        ('Material',schematic_material(cfg).removeprefix('Material '),schematic_material(y).removeprefix('Material ')),
        ('A / R (µm)',report_value(cfg.aperture_m,1e-6)+' / '+report_value(cfg.radius_m,1e-6),
                       report_value(y.aperture_m,1e-6)+' / '+report_value(y.radius_m,1e-6)),
        ('d / L (µm)',report_value(cfg.bridge_thickness_m,1e-6)+' / '+report_value(cfg.structure_depth_m,1e-6),
                       report_value(y.bridge_thickness_m,1e-6)+' / '+report_value(y.structure_depth_m,1e-6)),
        ('Density (g/cm³)',report_value(cfg.density_g_cm3),report_value(y.density_g_cm3 if y.material_delta is None else None)),
        ('CRL length',alignment_length(alignment.x_lens.lens_length_m),alignment_length(alignment.y_lens.lens_length_m)),
    ]
    for i,(label,x_value,y_value) in enumerate(geometry):
        height=.808-.0153*i
        fig.text(.538,height,label,fontsize=7.9,color='#43576e')
        fig.text(.742,height,x_value,ha='right',fontsize=7.9,weight='semibold',color='#13273e')
        fig.text(.935,height,y_value,ha='right',fontsize=7.9,weight='semibold',color='#13273e')

    rows=[('Photon energy',report_value(cfg.energy_kev,unit='keV'))]
    if cfg.source_distance_m is not None:
        rows.append(('Source to x CRL entrance',report_value(cfg.source_distance_m,unit='m')))
    if cfg.source_type=='gaussian':
        rows.append(('Source FWHM x / y',report_values((cfg.source_fwhm_x_m,cfg.source_fwhm_y_m),1e-6,'µm')))
    rows.extend((('Gap · x exit to y entrance',alignment_length(alignment.gap_m)),
                 ('Total span',alignment_length(result.lens_length_m))))
    for i,(label,value) in enumerate(rows):
        height=.808-.0153*i
        fig.text(.058,height,label,fontsize=8,color='#43576e')
        fig.text(.49,height,value,ha='right',fontsize=8,weight='semibold',color='#13273e')


def report_crossed_measurements(fig,result,alignment):
    separation=alignment.image_separation_m
    if alignment.aligned:
        predicted=[('Common x / y image from y exit',alignment_length(alignment.common_image_m))]
    else:
        predicted=[('x image from y exit',alignment_length(alignment.image_x_m)),
                   ('y image from y exit',alignment_length(alignment.image_y_m))]
    predicted.append(('Image-plane separation',alignment_length(None if separation is None else abs(separation))))
    if not alignment.aligned and alignment.recommended_gap_m is not None:
        predicted.append(('Gap for a common image',alignment_length(alignment.recommended_gap_m)))
    if result.configuration.source_type!='plane':
        predicted.append(('Magnification',report_values(alignment.magnification_at_image)))
    zscale,zunit=axial_units(result)
    peak_label='Peak position (sampled)' if result.numerical_peak.sampled else 'Peak position (numerical)'
    measured=[(peak_label,report_value(result.peak_z_m,zscale,zunit)),
              ('Depth of focus x / y',report_depth_of_focus(result)),
              ('Transmission',report_value(result.transmission,.01,'%')),
              ('Equivalent aperture area',report_value(result.equivalent_aperture_area_m2,1e-12,'µm²'))]
    match=result.equal_fwhm_plane
    measured.append(('Equal-FWHM position from y exit',report_value(match.z_m,zscale,zunit) if match is not None else '—'))
    measured.append(('FWHM minimum positions x / y',report_values(result.sampled_fwhm_minimum_z_m,zscale,zunit)))
    if result.configuration.source_type=='gaussian':
        measured.append(('FWHM ratio (Gaussian/point)',report_values(result.source_broadening_factor)))
    report_metric_columns(fig,predicted,measured,'Paraxial image positions')
    status='Common downstream image plane' if alignment.aligned else 'No common downstream image plane'
    fig.text(.062,.602,status,fontsize=8.3,weight='semibold',
             color='#176d60' if alignment.aligned else '#8b512b')


def report_setup(fig,result,analytical):
    cfg=result.configuration
    source=[('Photon energy',report_value(cfg.energy_kev,unit='keV'))]
    if cfg.source_distance_m is not None:
        source.append(('Source-to-CRL distance',report_value(cfg.source_distance_m,unit='m')))
    if cfg.source_type=='gaussian':
        sizes=(cfg.source_fwhm_x_m,) if cfg.grid_dimension==1 else (cfg.source_fwhm_x_m,cfg.source_fwhm_y_m)
        source.append(('Source FWHM',report_values(sizes,1e-6,'µm')))
    source.extend((('CRL length',report_value(analytical.lens_length_m,1e-3,'mm')),
                   ('Maximum element count',str(analytical.max_element_count))))
    material=cfg.material_formula or 'Direct δ / β'
    if cfg.density_g_cm3 is not None:
        material+=' · '+report_value(cfg.density_g_cm3,unit='g/cm³')
    dimensions=[('Material / density',material),
                ('Geometric aperture A / apex radius R',report_value(cfg.aperture_m,1e-6)+' / '+report_value(cfg.radius_m,1e-6,'µm')),
                ('Web thickness d',report_value(cfg.bridge_thickness_m,1e-6,'µm'))]
    if cfg.geometry=='one-dimensional' and cfg.grid_dimension==2:
        dimensions.append(('Structure depth L',report_value(cfg.structure_depth_m,1e-6,'µm')))
    dimensions.append(('Element length p',report_value(analytical.element_length_m,1e-6,'µm')))
    for rows,left,right in ((source,.058,.450),(dimensions,.538,.935)):
        for i,(label,value) in enumerate(rows):
            y=.808-.0153*i
            fig.text(left,y,label,fontsize=8.15,color='#43576e')
            fig.text(right,y,value,fontsize=8.25,ha='right',weight='semibold',color='#13273e')


def report_footer(fig,result,device,elapsed,basis,reference):
    def mesh(shape,dx,dy):
        return (' × '.join(map(str,reversed(shape)))+' · Δx '+report_value(dx,1e-9,'nm')
                +(' · Δy '+report_value(dy,1e-9,'nm') if dy is not None else ''))
    cfg=result.configuration
    accuracy='Fast' if cfg.accuracy=='preview' else 'Accurate'
    fig.text(.054,.074,'Calculation settings',fontsize=9.6,weight='bold')
    runtime=f'{accuracy} · {device} · Run time {elapsed:.2f} s'
    fig.text(.054,.058,runtime,fontsize=7.45,color='#394d64')
    fig.text(.941,.058,basis,fontsize=7.1,color='#394d64',ha='right')
    propagation=(result.grid_shape,result.grid_step_m,result.grid_step_y_m)
    evaluation=(result.detector_shape,result.detector_step_m,result.detector_step_y_m)
    fig.text(.054,.044,'Propagation grid '+mesh(*propagation),fontsize=7.1,color='#394d64')
    if propagation!=evaluation:
        fig.text(.941,.044,'Field evaluation '+mesh(*evaluation),ha='right',fontsize=7.05,color='#394d64')
    region='manual bounds' if cfg.scan_range_m is not None else 'auto bounds'
    same_peak=result.crossed_alignment is None and math.isclose(reference,result.reference_metrics.max_intensity,rel_tol=1e-9)
    ref_label='peak at B' if same_peak else report_value(reference,unit='a.u.')
    fig.text(.054,.029,f'Axial scan {result.z_m.size} samples · Δz {axial_interval(result.scan_step_m)} · '
             f'{region}    |    Iref = {ref_label}',fontsize=7.2,color='#394d64')
    info_flags={'central_source_reference','convolution_model','mesh_window','crossed_alignment'}
    names={'detector_window':'evaluation window limit','detector_cap':'evaluation grid cap',
           'detector_boundary':'intensity reaches transverse boundary','sampling':'coarse FWHM sampling',
           'fwhm_missing':'unresolved FWHM','z_roi_boundary':'axial search boundary'}
    codes=[item.partition(':')[0] for item in result.diagnostics]
    diagnostics=[names.get(code,code.replace('_',' ')) for code in codes if code not in info_flags]
    if not result.numerical_peak.converged:diagnostics.append('peak search did not converge')
    if diagnostics:
        label='Flags: '+', '.join(dict.fromkeys(diagnostics))
        fig.text(.054,.015,textwrap.shorten(label,width=115,placeholder=' …'),fontsize=7,color='#8b512b')


def draw_report_qr(figure):
    from matplotlib.collections import PolyCollection

    # Fixed repository URL, QR version 3, error correction M.
    repository='https://github.com/mamichberdey/XPOSim'
    data=bytes.fromhex(
        'feba23fc17f5906eb9a6bb74e8b5dbadff2ec13d1507faaaafe00fb3009facd4be04'
        '73ad8bb358693bd9de9e34d0b0ea9b51fe8d17faad021635a7d29c446bb385b79854'
        '33d13724cf286cff0069c463fb8eeb105f5f10bade1fcdd48d206e94c36f04f046df'
        'e9f1a800'
    )
    size=29
    bits=np.unpackbits(np.frombuffer(data,dtype=np.uint8),count=size*size).reshape(size,size)
    cells=[[(x,y),(x+1,y),(x+1,y+1),(x,y+1)] for y,x in np.argwhere(bits)]
    ax=figure.add_axes([.857,.921,.090,.064])
    ax.add_collection(PolyCollection(cells,facecolors='black',edgecolors='none',antialiased=False))
    # Keep a four-module quiet zone around the code.
    ax.set(xlim=(-4,size+4),ylim=(size+4,-4),aspect='equal')
    ax.set_axis_off()
    figure.text(.902,.913,'GitHub',ha='center',fontsize=7.2,color='#355f85',url=repository)


def build_pdf_report(result,device,elapsed):
    """A single portrait A4 sheet tied to the last completed simulation."""
    from matplotlib.lines import Line2D
    cfg=result.configuration
    alignment=result.crossed_alignment
    analytical=alignment or xpos.analyze_crl(cfg)
    if alignment is None:
        planes=report_planes(result)
    else:
        planes,centered_on_image=crossed_report_planes(result)
    view=result.display_signal_bounds_m
    now=datetime.now(timezone.utc)
    basis='Plane-wave reference' if cfg.source_type=='plane' else 'Point-source reference'
    if cfg.source_type=='gaussian':
        reference_label='full Gaussian source' if alignment is not None else 'central point-source reference'
        basis=('Intensity convolution · '+reference_label if cfg.source_method=='convolution'
               else ' × '.join(map(str,result.source_mode_counts))+' source points · '+reference_label)
    stream=io.BytesIO()
    style={'font.family':'DejaVu Sans','pdf.fonttype':42,'figure.facecolor':'white','axes.facecolor':'white',
           'text.color':'#172b42','axes.labelcolor':'#263a51','xtick.color':'#394d64','ytick.color':'#394d64',
           'savefig.bbox':None,'savefig.facecolor':'white'}
    with plt.rc_context(style):
        fig=plt.figure(figsize=REPORT_PAGE)
        try:
            fig.text(.054,.964,'CRL simulation report',fontsize=21,weight='bold')
            fig.text(.054,.938,'Optical setup and focus analysis',fontsize=8.5,color='#3e5570')
            fig.text(.838,.962,f'XPOSim {xpos.__version__}  ·  {now:%Y-%m-%d %H:%M UTC}',ha='right',fontsize=7.8)
            draw_report_qr(fig)
            for y in (.905,.733,.587,.345,.086):
                fig.add_artist(Line2D([.054,.947],[y,y],transform=fig.transFigure,color='#c3cfdc',lw=.65))
            source_name={'point':'Point source','plane':'Plane wave','gaussian':'Gaussian source'}[cfg.source_type]
            fig.text(.054,.888,'Source and optics · '+source_name,fontsize=9.7,weight='bold')
            optical_schematic(cfg,analytical,ax=fig.add_axes([.054,.816,.481,.065]),show_values=False)
            if alignment is None:
                geometry_name='Point focus' if cfg.geometry=='two-dimensional' else 'Line focus'
                fig.text(.538,.888,'CRL element · '+geometry_name,fontsize=9.7,weight='bold')
                element_schematic(cfg,analytical,ax=fig.add_axes([.561,.815,.342,.068]),show_values=False)
                report_setup(fig,result,analytical)
                report_measurements(fig,result,analytical)
            else:
                fig.text(.585,.888,'x CRL element',fontsize=9.7,weight='bold')
                fig.text(.795,.888,'y CRL element',fontsize=9.7,weight='bold')
                element_schematic(cfg,alignment.x_lens,ax=fig.add_axes([.545,.819,.19,.063]),
                                  show_values=False,show_depth=True,focusing_axis='x')
                element_schematic(cfg.second_crl,alignment.y_lens,ax=fig.add_axes([.754,.819,.19,.063]),
                                  show_values=False,show_depth=True,focusing_axis='y')
                report_crossed_setup(fig,result,alignment)
                report_crossed_measurements(fig,result,alignment)
            fig.text(.054,.571,'Propagation along z',fontsize=11,weight='bold')
            if any(result.depth_of_focus_bounds_m):
                fig.text(.934,.571,'Shading: depth of focus',
                         ha='right',fontsize=7.2,color='#356a57')
            fig.text(.054,.330,'Transverse sections',fontsize=11,weight='bold')
            report_color_scale(fig)
            reference=max([result.intensity_reference,np.finfo(float).tiny,
                           *(plane.metrics.max_intensity for plane in planes if plane is not None)])
            report_scan(fig,result,planes,reference)
            if alignment is None:
                report_transverse(fig,result,planes,reference,view)
            else:
                report_crossed_transverse(fig,result,planes,reference,centered_on_image)
            report_footer(fig,result,device,elapsed,basis,reference)
            fig.savefig(stream,format='pdf',dpi=240,metadata={
                'Title':'CRL simulation report','Author':'XPOSim','CreationDate':now})
        finally:
            plt.close(fig)
    return stream.getvalue()


_report_directory=globals().get('_report_directory')

def download_report(_=None):
    global _report_directory
    result=state.get('result')
    if result is None or 'report_runtime' not in state:
        return
    report_button.disabled=True
    report_message.value='Preparing a print-ready A4 report…'
    try:
        data=build_pdf_report(result,*state['report_runtime'])
        filename='XPOSim_CRL_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.pdf'
        # A self-contained link also works in notebook frontends without Colab's download API.
        encoded=base64.b64encode(data).decode('ascii')
        report_message.value=(f'<a download="{filename}" href="data:application/pdf;base64,{encoded}">'
                              'Save PDF report</a>')
        if colab_output is not None:
            from google.colab import files as colab_files
            if _report_directory is None:
                _report_directory=tempfile.TemporaryDirectory(prefix='xposim-report-')
            path=Path(_report_directory.name)/filename
            path.write_bytes(data)
            colab_files.download(str(path))
    except Exception as exc:
        if not report_message.value.startswith('<a '):
            report_message.value='PDF export failed · '+html.escape(str(exc))
    finally:
        report_button.disabled=state.get('result') is None


compute_button.on_click(compute_focus)
report_button.on_click(download_report)
hardware_notice=widgets.HTML('<div style="font-size:12px;line-height:1.25;padding:1px 0">'+_hardware_help()+'</div>')
left=widgets.VBox([scan_heading,scan_image,scan_compact_image],layout=widgets.Layout(flex='1 1 470px',min_width='0',width='auto'))
right=widgets.VBox([plane_heading,plane_image,plane_compact_image],layout=widgets.Layout(flex='1 1 470px',min_width='0',width='auto'))
left.add_class('xposim-panel');right.add_class('xposim-panel')
results_panel=widgets.HBox([left,right],layout=widgets.Layout(width='100%',flex_flow='row wrap',align_items='stretch',gap='10px',display='none'))
details.layout.display='none'
toolbar=widgets.HBox([compute_button,status,theme_selector],layout=widgets.Layout(width='100%',align_items='center',gap='12px',margin='3px 0 0'))
toolbar.add_class('xposim-toolbar')
result_css=UI_CSS.replace('</style>', '''.xp-depth-readout {font-size:12px;font-weight:500}
.xp-panel-note {flex-basis:100%;font-size:12px;font-weight:400;color:var(--xp-secondary)}
.xp-source-widths {display:flex;flex-wrap:wrap;align-items:baseline;gap:4px 14px;padding:6px 9px;
 border:1px solid var(--xp-border);background:var(--xp-surface);margin:0 0 6px;font-size:13px}
</style>''')
result_style=widgets.HTML(result_css,layout=widgets.Layout(height='0px',margin='0',overflow='visible'))
explorer_root=widgets.VBox([result_style,hardware_notice,toolbar,results_notice,progress_view,summary,source_comparison,results_panel,plane_controls,details,report_panel],
                         layout=widgets.Layout(width='100%',max_width='1200px',margin='0 auto'))
explorer_root.add_class('xposim-explorer')
results_panel.add_class('xposim-results')
right.add_class('xposim-detector')
plane_controls.add_class('xposim-slider-below')
plane_controls.add_class('xposim-selection-controls')
summary.add_class('xposim-summary')
theme_selector.observe(lambda change:apply_theme(detect=True),names='value')
display(explorer_root)
apply_theme(refresh=False)
if not _first_explorer_initialization:
    status.value='<span style="font-size:12px">Parameters retained. Press Compute.</span>'


<details>
<summary>What the results mean</summary>

| Quantity | Physical meaning |
| --- | --- |
| Line, point and crossed focusing | A line-focus CRL focuses along x. An axisymmetric point-focus CRL focuses along x and y. A crossed pair uses one line-focus CRL per axis. The x stack comes first and the y stack is rotated by 90°. |
| Inter-stack gap and image-plane separation | The gap runs from the x CRL exit to the y CRL entrance. Both image positions use the y CRL exit as the axial origin. Auto-align finds the smallest nonnegative gap that puts both paraxial images in the same downstream plane. A manual gap can produce two distinct line-image planes. |
| Geometric aperture, radius of curvature and web thickness | The geometric aperture is the full width or diameter of the clear opening. The radius of curvature is specified at the apex of each surface. The web thickness is the minimum thickness of material between the two surfaces. |
| Structure depth and CRL length | Structure depth is the extent of a line-focus element along the unfocused y axis. Element length and CRL length are the axial dimensions of one element and the full stack, respectively. |
| Refractive index decrement and absorption index | The refractive index decrement describes refraction. The absorption index describes the material's contribution to attenuation. |
| Back focal distance and image distance | Back focal distance runs from the CRL exit to the paraxial focus of a plane wave. Image distance runs from the same exit to the predicted image plane for a finite source distance. |
| Maximum element count and transverse magnification | The element limit is the largest stack that places the paraxial focus of a plane wave beyond the CRL exit. Transverse magnification describes the ratio of a small image displacement to the corresponding source displacement along a focusing axis. A negative value indicates inversion. |
| Peak-intensity position | The axial distance from the CRL exit where the transverse intensity is highest. Crossed CRLs use the full source in the calculation region. A sampled result selects the strongest evaluated plane. For a single stack with Gaussian illumination the central point sets the reference. |
| FWHM and depth of focus | The full width at half maximum (FWHM) is measured between the half-maximum crossings around the intensity peak. Each focusing axis can have its own axial position of minimum FWHM. Shading shows where the FWHM stays within about 41% of its sampled minimum. |
| Equal-FWHM position | Axial distance from the y CRL exit where the full-source FWHMs along x and y agree within 0.1%. When several matches are resolved in the scan interval, the closest to the numerical peak is reported. |
| Transmission and web transmission | Transmission is the fraction of power entering the aperture that reaches the exit. Web transmission isolates attenuation through the central webs. |
| Effective aperture | The analytical width, or equivalent diameter for a point-focus lens, that characterizes throughput limited by the curved lens surfaces. The model excludes attenuation through the webs. |
| Equivalent aperture from transmission | The width, diameter or area of an ideal clear opening with the same transmitted fraction under uniform illumination of the entrance pupil. For crossed CRLs the area uses the rectangular entrance aperture of the x stack and the simulated transmission through both CRLs. |
| Rocking curve FWHM | The angular width of the transmission curve at half its peak value as the incident plane wave is tilted. |
| Finite-source broadening and transverse coherence length | Finite-source broadening compares the FWHM for the Gaussian source and its central point at the same plane. The coherence length is the separation between a centered pair of slits at half fringe visibility. |
| Gaussian source sampling | Intensity convolution shifts and sums copies of the central-point intensity response. For crossed CRLs, the kernel width comes from the beam displacement of off-axis test waves through both stacks. Point sampling also includes changes in the shape and transmission of each off-axis response. Its x and y counts are independent for crossed CRLs. A sampling flag means adjacent points may still be visible in the beam profile. |
| Relative intensity | The longitudinal and transverse maps share one reference peak taken over the scan and its reference plane. For two-axis focusing, the x–z and y–z maps use the samples nearest the optical axis in the perpendicular direction. |

</details>
